# 🏥 Pipeline de données — Prédiction du taux d'incidence
**Projet DataScientest / Liora — Direction de l'Actuariat Vie**

---

## Architecture : Schéma en étoile (Star Schema)

```
                              ┌────────────────────┐
                              │     dim_temps      │
                              │  PK: annee_mois    │
                              └─────────┬──────────┘
                                        │ annee_mois
                                        │
  ┌─────────────┐                       │                              ┌─────────────┐
  │ dim_geo_pop │                       ▼                              │   dim_csp   │
  │  PK: dept   ├──── dept ────►┌─────────────────────────┐◄─── dept ──┤  PK: dept   │
  └─────────────┘               │ fact_urgences           │            └─────────────┘
                                │                         │
                                │ FK: dept                │
                                │ FK: annee_mois          │
                                │                         │
                                │ taux_urgences_allergie  │
                                │ taux_urgences_asthme    │
                                │ taux_urgences_bronchio  │
                                │ taux_hosp_*             │
                                │ taux_sos_*              │
                                └───────────┬─────────────┘
                                            │ dept × annee_mois
                           ┌────────────────┼──────────────────┐
                           │                │                  │
                           ▼                ▼                  ▼
               ┌──────────────────┐ ┌──────────────┐ ┌──────────────────┐
               │    dim_meteo     │ │  dim_pollen  │ │ dim_qualite_air  │
               │ PK: dept×mois    │ │ PK: dept×mois│ │  PK: dept×mois   │
               └──────────────────┘ └──────────────┘ └──────────────────┘
```

> **Convention :** les flèches indiquent la direction de la jointure (FK → PK).  
> La table `fact_urgences` est la table centrale ; toutes les dimensions s'y joignent.

---

## Tables du modèle

| Table | Clé primaire (PK) | Granularité | Source | Lignes |
|---|---|---|---|---|
| `fact_urgences` | dept × annee_mois | Mois × Département | Santé Publique France | 6 912 |
| `dim_temps` | annee_mois | Mois | Calculé | 72 |
| `dim_meteo` | dept × annee_mois | Mois × Département | Météo France SYNOP | ~2 883 |
| `dim_qualite_air` | dept × annee_mois | Mois × Département (2021–2025) | AASQA / data.gouv.fr | ~2 518 |
| `dim_pollen` | dept × annee_mois | Mois × Département | RNSA | ~1 686 |
| `dim_geo_pop` | dept | Département | INSEE + Ameli | 96 |
| `dim_csp` | dept | Département | INSEE CSP | 96 |

---

## Flux de construction (dépendances)

```
  [Fichiers bruts]          [Fonctions build_*]         [Tables .parquet]
  ─────────────────         ──────────────────          ─────────────────

  (calculé)           ──►  build_dim_temps()       ──►  dim_temps.parquet
                                                           │
  allergie/asthme/    ──►  parse_urgences()         ──►  fact_urgences.parquet
  bronchiolite CSV                                         │
                                                           │
  synop_YYYY.csv.gz   ──►  build_station_dict()    ──►  stations_dept.parquet
       +              └──► build_dim_meteo()        ──►  dim_meteo.parquet
  liste stations                                           │
                                                           │
  insee_demographie   ──►  build_dim_geo_pop()      ──►  dim_geo_pop.parquet
  + urbanisation                                           │
  + ameli_*.xls                                            │
                                                           │
  insee_csp.xlsx      ──►  build_dim_csp()          ──►  dim_csp.parquet
                                                           │
  AASQA/FR_E2_*.csv   ──►  build_dim_qualite_air()  ──►  dim_qualite_air.parquet
                                                           │
  BDD_daily/*.xls     ──►  build_dim_pollen()        ──►  dim_pollen.parquet
                                                           │
  (toutes les tables) ──►  build_model_view()        ──►  df_model.parquet ✅
```


---
## 0. Imports & configuration

In [11]:
# ── Import les bibliothèques nécessaires ─────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
from pathlib import Path
import gzip
import reverse_geocoder as rg
import re

In [12]:
# ── Dossiers ─────────────────────────────────────────────────────────────────
RAW_DIR   = Path("data/raw")
TABLES_DIR = Path("data/tables")   # une table = un fichier .parquet
for d in [RAW_DIR, TABLES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── Paramètres globaux ───────────────────────────────────────────────────────
ANNEE_DEBUT = 2020
ANNEE_FIN   = 2025

DEPTS = [f"{i:02d}" for i in range(1, 96) if i != 20] + ["2A", "2B"]
# Décommentez pour inclure les DOM :
# DEPTS += ["971", "972", "973", "974", "976"]

print(f"✅ Config : {len(DEPTS)} départements | {ANNEE_DEBUT}–{ANNEE_FIN}")
print(f"   Dossier tables : {TABLES_DIR.resolve()}")

✅ Config : 96 départements | 2020–2025
   Dossier tables : /Users/siranh/Documents/Data Scientest/projet_liora/data/tables


---
## 1. ⏱️ `dim_temps` — Dimension temporelle

Entièrement calculée, aucun fichier à télécharger.

In [13]:
def build_dim_temps() -> pd.DataFrame:
    """
    Génère la dimension temporelle : une ligne par mois.
    Clé primaire : annee_mois (format 'YYYY-MM')
    """
    mois_range = pd.period_range(start=f"{ANNEE_DEBUT}-01",
                                  end=f"{ANNEE_FIN}-12", freq="M")
    df = pd.DataFrame({"annee_mois": mois_range.astype(str)})

    df["annee"]      = mois_range.year
    df["mois"]       = mois_range.month
    df["trimestre"]  = ((mois_range.month - 1) // 3) + 1
    df["semestre"]   = ((mois_range.month - 1) // 6) + 1

    # Encodage cyclique (évite la rupture déc → jan dans les modèles linéaires)
    df["sin_mois"]   = np.sin(2 * np.pi * df["mois"] / 12).round(6)
    df["cos_mois"]   = np.cos(2 * np.pi * df["mois"] / 12).round(6)

    # Indicateurs saisonniers (hémisphère nord)
    df["est_hiver"]     = df["mois"].isin([12, 1, 2]).astype(int)
    df["est_printemps"] = df["mois"].isin([3, 4, 5]).astype(int)
    df["est_ete"]       = df["mois"].isin([6, 7, 8]).astype(int)
    df["est_automne"]   = df["mois"].isin([9, 10, 11]).astype(int)

    # Saison pollinique (indicatif France métropolitaine)
    df["saison_pollen"] = df["mois"].isin([3, 4, 5, 6]).astype(int)

    # Flag Covid (peut influer sur les passages aux urgences)
    df["flag_covid"] = (
        ((df["annee"] == 2020) & (df["mois"] >= 3)) |
        (df["annee"] == 2021) |
        ((df["annee"] == 2022) & (df["mois"] <= 3))
    ).astype(int)

    print(f"dim_temps : {df.shape[0]} mois ({df['annee_mois'].iloc[0]} → {df['annee_mois'].iloc[-1]})")
    print(f"Colonnes  : {list(df.columns)}")
    return df


dim_temps = build_dim_temps()
dim_temps.to_parquet(TABLES_DIR / "dim_temps.parquet", index=False)
print(f"\n✅ Sauvegardé → {TABLES_DIR / 'dim_temps.parquet'}")
dim_temps.head(8)

dim_temps : 72 mois (2020-01 → 2025-12)
Colonnes  : ['annee_mois', 'annee', 'mois', 'trimestre', 'semestre', 'sin_mois', 'cos_mois', 'est_hiver', 'est_printemps', 'est_ete', 'est_automne', 'saison_pollen', 'flag_covid']

✅ Sauvegardé → data/tables/dim_temps.parquet


,annee_mois,annee,mois,trimestre,semestre,sin_mois,cos_mois,est_hiver,est_printemps,est_ete,est_automne,saison_pollen,flag_covid
0,2020-01,2020,1,1,1,0.500000,0.866025,1,0,0,0,0,0
1,2020-02,2020,2,1,1,0.866025,0.500000,1,0,0,0,0,0
2,2020-03,2020,3,1,1,1.000000,0.000000,0,1,0,0,1,1
3,2020-04,2020,4,2,1,0.866025,-0.500000,0,1,0,0,1,1
4,2020-05,2020,5,2,1,0.500000,-0.866025,0,1,0,0,1,1
5,2020-06,2020,6,2,1,0.000000,-1.000000,0,0,1,0,1,1
6,2020-07,2020,7,3,2,-0.500000,-0.866025,0,0,1,0,0,1
7,2020-08,2020,8,3,2,-0.866025,-0.500000,0,0,1,0,0,1


---
## 2. 🎯 `fact_urgences` — Table de faits

**Source :** Santé Publique France — Odisse  

### Fichiers à télécharger → `data/raw/`
| Pathologie | URL | Nom fichier |
|---|---|---|
| Allergie | https://odisse.santepubliquefrance.fr/explore/dataset/allergie-passages-aux-urgences-et-actes-sos-medecins-dep/export/ | `allergie_urgences.csv` |
| Asthme | https://odisse.santepubliquefrance.fr/explore/dataset/asthme-passages-aux-urgences-et-actes-sos-medecins-dep/export/ | `asthme_urgences.csv` |
| Bronchiolite | https://odisse.santepubliquefrance.fr/explore/dataset/bronchiolite-passages-aux-urgences-et-actes-sos-medecins-departement/export/ | `bronchiolite_urgences.csv` |

> **Format export :** choisir **CSV avec séparateur `;`**

In [14]:
for fname in ["asthme_urgences.csv", "bronchiolite_urgences.csv"]:
    df = pd.read_csv(f"data/raw/{fname}", sep=",", nrows=1, dtype=str)
    print(f"\n{fname} :")
    for col in df.columns:
        print(f"  '{col}'")


asthme_urgences.csv :
  '1er jour de la semaine'
  'Semaine'
  'Département Code'
  'Département'
  'Classe d'âge'
  'Taux de passages aux urgences pour asthme'
  'Taux d'hospitalisations après passages aux urgences pour asthme'
  'Taux d'actes médicaux SOS médecins pour asthme'
  'Région Code'
  'Région'

bronchiolite_urgences.csv :
  '1er jour de la semaine'
  'Semaine'
  'Département Code'
  'Département'
  'Classe d'âge'
  'Taux de passages aux urgences pour bronchiolite'
  'Taux d'hospitalisations après passages aux urgences pour bronchiolite'
  'Taux d'actes médicaux SOS médecins pour bronchiolite'
  'Région Code'
  'Région'


In [15]:
df = pd.read_csv("data/raw/bronchiolite_urgences.csv", sep=",", dtype=str)
df = df.iloc[:, [0, 2, 4, 5, 6, 7]]
df.columns = ["date", "dept", "classe_age","taux_urgences", "taux_hosp", "taux_sos"]

# Voir toutes les valeurs uniques de classe_age
print(df["classe_age"].unique())

# il n'y a que 0 an dans la colonne classe_age, on peut filtrer dessus pour ne garder que les lignes "Tous âges"

['0 an']


In [16]:
def parse_urgences(pathologie: str, filepath: Path) -> pd.DataFrame:
    """
    Charge et nettoie un fichier CSV de Santé Publique France.
 
    PARAMÈTRES :
    ─────────────
    pathologie : "allergie", "asthme" ou "bronchiolite"
    filepath   : chemin vers le fichier CSV brut
 
    RETOUR :
    ─────────
    DataFrame avec colonnes :
        dept                        → code département (ex: "75")
        annee_mois                  → période (ex: "2021-03")
        taux_urgences_{pathologie}  → passages aux urgences pour 100k hab
        taux_hosp_{pathologie}      → hospitalisations après urgences
        taux_sos_{pathologie}       → actes SOS Médecins
 
    STRATÉGIE :
    ─────────────
    On utilise les POSITIONS des colonnes (iloc) plutôt que leurs noms,
    car les fichiers mélangent guillemets simples et doubles ce qui
    rend toute comparaison de chaînes instable.
 
    Structure commune aux 3 fichiers :
        [0] 1er jour de la semaine  ← date
        [1] Semaine
        [2] Département Code        ← dept
        [3] Département
        [4] Classe d'âge            ← filtre age
        [5] Taux urgences           ← Y principal
        [6] Taux hospitalisations   ← Y secondaire
        [7] Taux SOS médecins       ← Y tertiaire
        [8] Région Code
        [9] Région
    """
 
    # ── ÉTAPE 1 : Chargement ─────────────────────────────────────────────────
    # dtype=str → tout en texte d'abord pour éviter les erreurs de type mixte
    df = pd.read_csv(filepath, sep=",", encoding="utf-8",
                     dtype=str, low_memory=False)
    print(f"\n[{pathologie.upper()}]")
    print(f"  Lignes brutes       : {len(df):,}")
 
    # ── ÉTAPE 2 : Sélection et renommage par position ────────────────────────
    # On prend uniquement les 8 premières colonnes utiles
    # et on leur donne des noms clairs tout de suite
    df = df.iloc[:, [0, 2, 4, 5, 6, 7]].copy()
    df.columns = [
        "date",
        "dept",
        "classe_age",
        f"taux_urgences_{pathologie}",
        f"taux_hosp_{pathologie}",
        f"taux_sos_{pathologie}",
    ]
 
    # ── ÉTAPE 3 : Filtrage sur la classe d'âge globale ───────────────────────
    # Allergie & Asthme  → "Tous âges"  (population complète)
    # Bronchiolite       → "0 an"       (nourrissons uniquement, pas de "Tous âges")
    # On garde les deux valeurs possibles avec isin()
    valeurs_age = df["classe_age"].unique().tolist()
    print(f"  Classes d'âge dispo : {valeurs_age}")
 
    valeurs_valides = ["Tous âges", "0 an"]
    df = df[df["classe_age"].isin(valeurs_valides)].copy()
    print(f"  Après filtre âge    : {len(df):,} lignes "
          f"({df['classe_age'].unique().tolist()})")
 
    # ── ÉTAPE 4 : Parsing de la date ─────────────────────────────────────────
    # La colonne contient le lundi de chaque semaine : "2020-03-30"
    # On la convertit pour extraire l'année et le mois
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    nb_invalides = df["date"].isna().sum()
    if nb_invalides > 0:
        print(f"  ⚠️  {nb_invalides} dates invalides supprimées")
    df = df.dropna(subset=["date"])
 
    # annee_mois = clé de jointure avec toutes les autres tables
    # Ex : semaines S13, S14, S15 de mars → toutes donnent "2020-03"
    df["annee_mois"] = df["date"].dt.to_period("M").astype(str)
    df["annee"]      = df["date"].dt.year
 
    # ── ÉTAPE 5 : Filtrage temporel 2020–2025 ────────────────────────────────
    avant = len(df)
    df = df[(df["annee"] >= ANNEE_DEBUT) & (df["annee"] <= ANNEE_FIN)]
    print(f"  Filtrage {ANNEE_DEBUT}–{ANNEE_FIN}      : {avant:,} → {len(df):,} lignes")
 
    # ── ÉTAPE 6 : Nettoyage du code département ───────────────────────────────
    # "9" → "09" | " 68" → "68" | "2a" → "2A"
    df["dept"] = (
        df["dept"]
        .astype(str)
        .str.strip()
        .str.upper()
        .str.zfill(2)
    )
    avant = len(df)
    df = df[df["dept"].isin(DEPTS)]
    print(f"  Filtrage depts      : {avant:,} → {len(df):,} lignes")
 
    # ── ÉTAPE 7 : Conversion numérique des taux ───────────────────────────────
    # Les valeurs étaient en texte (dtype=str au chargement)
    # errors="coerce" → les "-" ou "n/a" deviennent NaN
    cols_taux = [f"taux_urgences_{pathologie}",
                 f"taux_hosp_{pathologie}",
                 f"taux_sos_{pathologie}"]
    for col in cols_taux:
        df[col] = pd.to_numeric(df[col], errors="coerce")
 
    # ── ÉTAPE 8 : Agrégation hebdomadaire → mensuelle ─────────────────────────
    # Les données sont hebdomadaires (1 ligne par semaine × département)
    # On prend la MOYENNE des semaines du mois car ce sont des TAUX
    # (pas des comptages → la somme n'aurait pas de sens)
    df_agg = (
        df.groupby(["dept", "annee_mois"])[cols_taux]
        .mean()
        .round(2)
        .reset_index()
    )
 
    print(f"  ✅ Résultat         : {df_agg.shape[0]:,} lignes × {df_agg.shape[1]} colonnes")
    print(f"  Période             : {df_agg['annee_mois'].min()} → {df_agg['annee_mois'].max()}")
    print(f"  Départements        : {df_agg['dept'].nunique()}")
    return df_agg
 
 
# ══════════════════════════════════════════════════════════════════════════════
# CHARGEMENT DES 3 PATHOLOGIES
# ══════════════════════════════════════════════════════════════════════════════
 
FICHIERS_URGENCES = {
    "allergie":     "allergie_urgences.csv",
    "asthme":       "asthme_urgences.csv",
    "bronchiolite": "bronchiolite_urgences.csv",
}
 
dfs_patho = {}
for patho, fname in FICHIERS_URGENCES.items():
    fpath = RAW_DIR / fname
    if fpath.exists():
        dfs_patho[patho] = parse_urgences(patho, fpath)
    else:
        print(f"⚠️  Fichier manquant : {fname}")
 
 
# ══════════════════════════════════════════════════════════════════════════════
# FUSION EN UNE SEULE TABLE fact_urgences
# ══════════════════════════════════════════════════════════════════════════════
 
if dfs_patho:
 
    # Squelette complet : 96 depts × 72 mois = 6 912 lignes garanties
    # Même si un département n'a pas de données → ligne présente avec NaN
    dim_t = pd.read_parquet(TABLES_DIR / "dim_temps.parquet")[["annee_mois"]]
    skeleton = pd.MultiIndex.from_product(
        [DEPTS, dim_t["annee_mois"]],
        names=["dept", "annee_mois"]
    ).to_frame(index=False)
 
    # Jointure de chaque pathologie sur le squelette
    fact = skeleton.copy()
    for patho, df_tmp in dfs_patho.items():
        fact = fact.merge(df_tmp, on=["dept", "annee_mois"], how="left")
 
    # Sauvegarde
    fact.to_parquet(TABLES_DIR / "fact_urgences.parquet", index=False)
    print(f"\n✅ fact_urgences sauvegardé")
    print(f"   {fact.shape[0]:,} lignes × {fact.shape[1]} colonnes")
    display(fact.head(10))
    
    # Rapport de couverture
    print("\n📊 Couverture finale :")
    for col in [c for c in fact.columns if c.startswith("taux_")]:
        pct = fact[col].notna().mean() * 100
        statut = "✅" if pct > 80 else "⚠️ "
        barre = "█" * int(pct // 10) + "░" * (10 - int(pct // 10))
        print(f"  {statut} {col:<40} {barre} {pct:.1f}%")
    


[ALLERGIE]
  Lignes brutes       : 136,864
  Classes d'âge dispo : ['65 ans ou plus', 'Tous âges', '00-14 ans', '15-64 ans']
  Après filtre âge    : 34,216 lignes (['Tous âges'])
  Filtrage 2020–2025      : 34,216 → 32,552 lignes
  Filtrage depts      : 32,552 → 30,048 lignes
  ✅ Résultat         : 6,912 lignes × 5 colonnes
  Période             : 2020-01 → 2025-12
  Départements        : 96

[ASTHME]
  Lignes brutes       : 136,864
  Classes d'âge dispo : ['00-14 ans', '15-64 ans', '65 ans ou plus', 'Tous âges']
  Après filtre âge    : 34,216 lignes (['Tous âges'])
  Filtrage 2020–2025      : 34,216 → 32,552 lignes
  Filtrage depts      : 32,552 → 30,048 lignes
  ✅ Résultat         : 6,912 lignes × 5 colonnes
  Période             : 2020-01 → 2025-12
  Départements        : 96

[BRONCHIOLITE]
  Lignes brutes       : 34,216
  Classes d'âge dispo : ['0 an']
  Après filtre âge    : 34,216 lignes (['0 an'])
  Filtrage 2020–2025      : 34,216 → 32,552 lignes
  Filtrage depts      : 32,552

,dept,annee_mois,taux_urgences_allergie,taux_hosp_allergie,taux_sos_allergie,taux_urgences_asthme,taux_hosp_asthme,taux_sos_asthme,taux_urgences_bronchiolite,taux_hosp_bronchiolite,taux_sos_bronchiolite
0,01,2020-01,815.39,245.12,NaN,402.03,878.26,NaN,23287.66,44861.11,NaN
1,01,2020-02,722.06,228.66,NaN,462.67,437.93,NaN,10169.56,25875.35,NaN
2,01,2020-03,377.48,193.97,NaN,805.16,841.07,NaN,9939.19,14230.77,NaN
3,01,2020-04,500.03,98.04,NaN,574.89,863.81,NaN,0.00,0.00,NaN
4,01,2020-05,568.77,311.94,NaN,522.41,552.69,NaN,0.00,0.00,NaN
5,01,2020-06,796.83,585.61,NaN,272.48,368.71,NaN,2222.22,12500.00,NaN
6,01,2020-07,1211.83,323.65,NaN,345.01,398.95,NaN,2380.95,4166.67,NaN
7,01,2020-08,1335.92,312.04,NaN,343.45,514.11,NaN,833.33,0.00,NaN
8,01,2020-09,842.45,284.92,NaN,381.14,627.62,NaN,3010.88,9166.67,NaN
9,01,2020-10,652.28,300.14,NaN,461.62,594.91,NaN,3229.81,2777.78,NaN



📊 Couverture finale :
  ✅ taux_urgences_allergie                   ██████████ 100.0%
  ✅ taux_hosp_allergie                       █████████░ 99.9%
  ⚠️  taux_sos_allergie                        ████░░░░░░ 46.5%
  ✅ taux_urgences_asthme                     ██████████ 100.0%
  ✅ taux_hosp_asthme                         █████████░ 99.9%
  ⚠️  taux_sos_asthme                          ████░░░░░░ 46.5%
  ✅ taux_urgences_bronchiolite               █████████░ 100.0%
  ✅ taux_hosp_bronchiolite                   █████████░ 96.6%
  ⚠️  taux_sos_bronchiolite                    ████░░░░░░ 46.4%


---
## 3. 🌡️ `dim_meteo` — Dimension météorologique

**Source :** Météo France — Archives SYNOP (data.gouv.fr)  

### Fichiers à télécharger → `data/raw/synop/`
URL : https://www.data.gouv.fr/datasets/archive-synop-omm

Pour chaque année de 2020 à 2025, téléchargez le fichier `synop.YYYY.csv.gz`  
(ex : `synop.2020.csv.gz`, `synop.2021.csv.gz`, ...)

Téléchargez aussi la **liste des stations** :  
`data/raw/liste-stations-synop.csv`  

Une autre source des données :
URL : https://donneespubliques.meteofrance.fr/?fond=produit&id_produit=115&id_rubrique=38

In [17]:
# Vérifier la liste des stations
df_stations = pd.read_csv(
    "data/raw/synop/observations-liste-stations-synop-omm-20250710.csv", sep=";", nrows=5)
print("=== STATIONS ===")
print(df_stations.columns.tolist())
print(df_stations.head())

# Vérifier un fichier SYNOP
df_synop = pd.read_csv("data/raw/synop/synop_2020.csv.gz",
                       sep=";", compression="gzip", nrows=5)
print("\n=== SYNOP 2020 ===")
print(df_synop.columns.tolist())
display(df_synop.head())

=== STATIONS ===
['ID', 'Nom', 'Latitude', 'Longitude', 'Altitude']
     ID              Nom   Latitude  Longitude  Altitude
0  7005        ABBEVILLE  50.136000   1.834000        69
1  7015    LILLE-LESQUIN  50.570000   3.097500        47
2  7020  PTE DE LA HAGUE  49.725167  -1.939833         6
3  7027   CAEN-CARPIQUET  49.180000  -0.456167        67
4  7037       ROUEN-BOOS  49.383000   1.181667       151

=== SYNOP 2020 ===
['lat', 'lon', 'geo_id_wmo', 'geo_id_wigos', 'name', 'reference_time', 'insert_time', 'validity_time', 'pmer', 'tend', 'cod_tend', 'dd', 'ff', 't', 'td', 'u', 'vv', 'ww', 'w1', 'w2', 'n', 'nbas', 'hbas', 'cl', 'cm', 'ch', 'pres', 'niv_bar', 'geop', 'tend24', 'tn12', 'tn24', 'tx12', 'tx24', 'tminsol', 'sw', 'tw', 'raf10', 'rafper', 'per', 'etat_sol', 'ht_neige', 'ssfrai', 'perssfrai', 'rr1', 'rr3', 'rr6', 'rr12', 'rr24', 'phenspe1', 'phenspe2', 'phenspe3', 'phenspe4', 'nnuage1', 'ctype1', 'hnuage1', 'nnuage2', 'ctype2', 'hnuage2', 'nnuage3', 'ctype3', 'hnuage3', 'n

,lat,lon,geo_id_wmo,geo_id_wigos,name,reference_time,insert_time,validity_time,pmer,tend,...,hnuage1,nnuage2,ctype2,hnuage2,nnuage3,ctype3,hnuage3,nnuage4,ctype4,hnuage4
0,50.136000,1.834000,7005,0-20000-0-07005,ABBEVILLE,NaN,NaN,2020-01-01T00:00:00Z,103180.0,-80,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,48.445500,0.110167,7139,0-20000-0-07139,ALENCON,NaN,NaN,2020-01-01T00:00:00Z,103180.0,-30,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,49.725167,-1.939833,7020,0-20000-0-07020,PTE DE LA HAGUE,NaN,NaN,2020-01-01T00:00:00Z,102870.0,-70,...,600.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,48.324667,4.020000,7168,0-20000-0-07168,TROYES-BARBEREY,NaN,NaN,2020-01-01T00:00:00Z,103380.0,100,...,90.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,44.581167,4.733000,7577,0-20000-0-07577,MONTELIMAR,NaN,NaN,2020-01-01T00:00:00Z,103200.0,-50,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [18]:
def build_station_dict() -> dict:
    """
    Construit le dictionnaire {station_ID → code_dept} à partir de la liste
    des stations SYNOP OMM et d'un reverse geocoding (lat/lon → département).

    Sauvegarde aussi data/tables/stations_dept.parquet pour réutilisation.

    Retourne : dict { int(station_id): str(dept_code) }
    """
    stations_path = RAW_DIR / "synop" / "observations-liste-stations-synop-omm-20250710.csv"
    if not stations_path.exists():
        print(f"⚠️  Liste stations introuvable : {stations_path}")
        return {}

    df_stations = pd.read_csv(stations_path, sep=";")
    print(f"Stations totales : {len(df_stations)}")

    # Reverse geocoding : lat/lon → département
    coords = list(zip(df_stations["Latitude"], df_stations["Longitude"]))
    resultats = rg.search(coords)
    df_stations["dept_nom"] = [r["admin2"] for r in resultats]
    df_stations["pays"]     = [r["cc"]     for r in resultats]

    # Garder uniquement France métropolitaine
    df_fr = df_stations[df_stations["pays"] == "FR"].copy()
    print(f"Stations France  : {len(df_fr)}")

    # Nom département (reverse geocoder) → code INSEE
    DEPT_NOM_TO_CODE = {
        "Departement de l'Ain": "01", "Departement de l'Aisne": "02",
        "Departement de l'Allier": "03", "Departement des Hautes-Alpes": "05",
        "Departement des Alpes-Maritimes": "06", "Departement de l'Ardeche": "07",
        "Departement des Ardennes": "08", "Departement de l'Ariege": "09",
        "Departement de l'Aube": "10", "Departement de l'Aude": "11",
        "Departement de l'Aveyron": "12", "Departement des Bouches-du-Rhone": "13",
        "Departement du Calvados": "14", "Departement du Cantal": "15",
        "Departement de la Charente": "16", "Departement de la Charente-Maritime": "17",
        "Departement du Cher": "18", "Departement de la Correze": "19",
        "Departement de la Cote-d'Or": "21", "Departement des Cotes-d'Armor": "22",
        "Departement de la Creuse": "23", "Departement de la Dordogne": "24",
        "Departement du Doubs": "25", "Departement de la Drome": "26",
        "Departement de l'Eure": "27", "Departement d'Eure-et-Loir": "28",
        "Departement du Finistere": "29", "Departement du Gard": "30",
        "Departement de la Haute-Garonne": "31", "Departement du Gers": "32",
        "Departement de la Gironde": "33", "Departement de l'Herault": "34",
        "Departement d'Ille-et-Vilaine": "35", "Departement de l'Indre": "36",
        "Departement d'Indre-et-Loire": "37", "Departement de l'Isere": "38",
        "Departement du Jura": "39", "Departement des Landes": "40",
        "Departement du Loir-et-Cher": "41", "Departement de la Loire": "42",
        "Departement de la Haute-Loire": "43", "Departement de la Loire-Atlantique": "44",
        "Departement du Loiret": "45", "Departement du Lot": "46",
        "Departement du Lot-et-Garonne": "47", "Departement de la Lozere": "48",
        "Departement du Maine-et-Loire": "49", "Departement de la Manche": "50",
        "Departement de la Marne": "51", "Departement de la Haute-Marne": "52",
        "Departement de la Mayenne": "53", "Departement de Meurthe-et-Moselle": "54",
        "Departement de la Meuse": "55", "Departement du Morbihan": "56",
        "Departement de la Moselle": "57", "Departement de la Nievre": "58",
        "Departement du Nord": "59", "Departement de l'Oise": "60",
        "Departement de l'Orne": "61", "Departement du Pas-de-Calais": "62",
        "Departement du Puy-de-Dome": "63", "Departement des Pyrenees-Atlantiques": "64",
        "Departement des Hautes-Pyrenees": "65", "Departement des Pyrenees-Orientales": "66",
        "Departement du Bas-Rhin": "67", "Departement du Haut-Rhin": "68",
        "Departement du Rhone": "69", "Departement de la Haute-Saone": "70",
        "Departement de Saone-et-Loire": "71", "Departement de la Sarthe": "72",
        "Departement de la Savoie": "73", "Departement de la Haute-Savoie": "74",
        "Departement de Paris": "75", "Departement de la Seine-Maritime": "76",
        "Departement de Seine-et-Marne": "77", "Departement des Yvelines": "78",
        "Departement des Deux-Sevres": "79", "Departement de la Somme": "80",
        "Departement du Tarn": "81", "Departement du Tarn-et-Garonne": "82",
        "Departement du Var": "83", "Departement du Vaucluse": "84",
        "Departement de la Vendee": "85", "Departement de la Vienne": "86",
        "Departement de la Haute-Vienne": "87", "Departement des Vosges": "88",
        "Departement de l'Yonne": "89", "Departement du Territoire de Belfort": "90",
        "Departement de l'Essonne": "91", "Departement des Hauts-de-Seine": "92",
        "Departement de la Seine-Saint-Denis": "93", "Departement du Val-de-Marne": "94",
        "Departement du Val-d'Oise": "95",
        "Departement de la Corse-du-Sud": "2A", "Departement de la Haute-Corse": "2B",
    }

    df_fr["dept"] = df_fr["dept_nom"].map(DEPT_NOM_TO_CODE)

    # Diagnostics
    sans_code = df_fr[df_fr["dept"].isna()][["ID", "Nom", "dept_nom"]]
    if len(sans_code):
        print(f"⚠️  {len(sans_code)} stations sans correspondance département :")
        print(sans_code.to_string())
    else:
        print("✅ Toutes les stations mappées")

    df_fr = df_fr.dropna(subset=["dept"])

    # Sauvegarder pour les runs futurs
    df_fr[["ID", "Nom", "dept"]].to_parquet(
        TABLES_DIR / "stations_dept.parquet", index=False
    )
    print(f"✅ Sauvegardé → data/tables/stations_dept.parquet ({len(df_fr)} stations)")

    return dict(zip(df_fr["ID"], df_fr["dept"]))


station_to_dept = build_station_dict()


Stations totales : 62
Loading formatted geocoded file...
Stations France  : 42
✅ Toutes les stations mappées
✅ Sauvegardé → data/tables/stations_dept.parquet (42 stations)


In [19]:
df_stations = pd.read_csv(
    "data/raw/synop/observations-liste-stations-synop-omm-20250710.csv",
    sep=";"
)

coords = list(zip(df_stations["Latitude"], df_stations["Longitude"]))
resultats = rg.search(coords)

df_stations["dept_nom"] = [r["admin2"] for r in resultats]
df_stations["pays"] = [r["cc"] for r in resultats]

# Voir les stations hors France
hors_france = df_stations[df_stations["pays"] != "FR"]
print(f"Stations hors France ({len(hors_france)}) :")
print(hors_france[["ID", "Nom", "Latitude", "Longitude", "pays"]].to_string())

Stations hors France (20) :
       ID                  Nom   Latitude   Longitude pays
42  61968           GLORIEUSES -11.582667   47.289667   MG
43  61970         JUAN DE NOVA -17.054667   42.712000   MG
44  61972               EUROPA -22.344167   40.340667   MG
45  61976             TROMELIN -15.887667   54.520667   MG
46  61980      GILLOT-AEROPORT -20.892500   55.528667   RE
47  61996   NOUVELLE AMSTERDAM -37.795167   77.569167   TF
48  61997               CROZET -46.432500   51.856667   TF
49  61998            KERGUELEN -49.352333   70.243333   TF
50  67005             PAMANDZI -12.805500   45.282833   YT
51  71805            ST-PIERRE  46.766333  -56.179167   PM
52  78890    LA DESIRADE METEO  16.335000  -61.004000   GP
53  78894  ST-BARTHELEMY METEO  17.901500  -62.852167   BL
54  78897       LE RAIZET AERO  16.264000  -61.516333   GP
55  78922      TRINITE-CARAVEL  14.774500  -60.875333   MQ
56  78925        LAMENTIN-AERO  14.595333  -60.995667   MQ
57  81401        SAINT LAURE

In [20]:
# Le dictionnaire DEPT_NOM_TO_CODE et la sauvegarde sont maintenant
# encapsulés dans build_station_dict() (cellule précédente).
# station_to_dept est directement disponible après l'appel ci-dessus.
print(f"✅ {len(station_to_dept)} stations disponibles")
print(dict(list(station_to_dept.items())[:5]))  # aperçu


✅ 42 stations disponibles
{7005: '80', 7015: '59', 7020: '50', 7027: '14', 7037: '76'}


In [21]:
# Vérification : charger depuis le parquet sauvegardé (idempotent)
_df_check = pd.read_parquet(TABLES_DIR / "stations_dept.parquet")
assert len(_df_check) > 0, "stations_dept.parquet vide !"
print(f"✅ stations_dept.parquet : {len(_df_check)} stations")


✅ stations_dept.parquet : 42 stations


Notre variable cible — les passages aux urgences — est elle-même mensuelle. Le modèle a besoin de capturer des dynamiques saisonnières fines, par exemple :

Mars 2021 : début de la saison pollinique + remontée des températures → pic de passages aux urgences pour allergie
Décembre 2021 : chute des températures → augmentation des bronchiolites

Avec des données annuelles, ces effets saisonniers seraient complètement lissés et le modèle perdrait l'essentiel de sa capacité prédictive.
En résumé : le fichier est annuel par commodité de stockage, mais les données qu'il contient sont horaires. On les agrège au mois pour correspondre à la granularité de nos autres tables.

In [22]:
def build_dim_meteo(station_to_dept: dict) -> pd.DataFrame:
    """
    Construit la table dim_meteo à partir des fichiers SYNOP annuels.

    STRATÉGIE :
    ───────────
    1. Charger chaque fichier SYNOP annuel (synop_2020.csv.gz, ...)
    2. Rattacher chaque mesure à un département via station_to_dept
    3. Agréger toutes les mesures de toutes les stations d'un même
       département sur le même mois → moyenne mensuelle par département
    4. Sauvegarder en parquet

    VARIABLES MÉTÉO PRODUITES :
    ───────────────────────────
    temp_moy    → température moyenne mensuelle (°C)
    temp_min    → température minimale du mois (°C)
    temp_max    → température maximale du mois (°C)
    humidite_moy→ humidité relative moyenne (%)
    vent_moy    → vitesse moyenne du vent (m/s)
    vent_max    → rafale maximale du mois (m/s)
    precip_total→ précipitations totales mensuelles (mm)
    
    NIVEAUX D'AGRÉGATION :
    ───────────────────────────
        Données brutes SYNOP
    (toutes les 3h × station)
                 ↓
    groupby(dept, annee_mois).mean()
                 ↓
             dim_meteo
    (1 ligne par département × mois)
    

    NOTE SUR LES UNITÉS SYNOP :
    ───────────────────────────
    - Température 't' : en Kelvin → on soustrait 273.15 pour avoir °C
    - Humidité 'u'    : déjà en %
    - Vent 'ff'       : en m/s
    - Précipitations 'rr1' ou 'rr3' : en mm

    CLÉ PRIMAIRE : dept × annee_mois
    """

    synop_dir = RAW_DIR / "synop"

    # ── ÉTAPE 1 : Charger et concaténer tous les fichiers annuels ────────────
    dfs_annuels = []

    for annee in range(ANNEE_DEBUT, ANNEE_FIN + 1):
        fpath = synop_dir / f"synop_{annee}.csv.gz"
        if not fpath.exists():
            print(f"  ⚠️  Manquant : synop_{annee}.csv.gz")
            continue

        print(f"  Chargement synop_{annee}...", end=" ")

        # Le fichier est volumineux → on charge uniquement les colonnes utiles
        # geo_id_wmo = ID de la station (pour rattacher au département)
        # validity_time = date/heure de la mesure
        # t   = température (Kelvin)
        # u   = humidité relative (%)
        # ff  = vitesse du vent (m/s)
        # fx  = rafale maximale (m/s)
        # rr1 = précipitations sur 1h (mm)
        cols_utiles = ["lat", "lon", "geo_id_wmo", "validity_time",
                       "t", "u", "ff", "fx", "rr1"]

        df = pd.read_csv(
            fpath,
            sep=";",
            compression="gzip",
            usecols=lambda c: c in cols_utiles,  # ne charge que les colonnes utiles
            low_memory=False
        )
        print(f"{len(df):,} lignes")
        dfs_annuels.append(df)

    if not dfs_annuels:
        print("❌ Aucun fichier SYNOP trouvé dans data/raw/synop/")
        return pd.DataFrame()

    # Concaténer toutes les années en un seul DataFrame
    df = pd.concat(dfs_annuels, ignore_index=True)
    print(f"\n  Total brut : {len(df):,} lignes")

    # ── ÉTAPE 2 : Rattachement station → département ─────────────────────────
    # geo_id_wmo contient l'ID numérique de la station
    # On le mappe vers le code département via notre dictionnaire
    df["geo_id_wmo"] = pd.to_numeric(df["geo_id_wmo"], errors="coerce")
    df["dept"] = df["geo_id_wmo"].map(station_to_dept)

    # Supprimer les mesures dont la station n'est pas dans notre dictionnaire
    # (stations hors France métropolitaine, stations inconnues)
    avant = len(df)
    df = df.dropna(subset=["dept"])
    print(f"  Après filtre stations FR : {avant:,} → {len(df):,} lignes")

    # ── ÉTAPE 3 : Parsing de la date ─────────────────────────────────────────
    # validity_time contient la date et l'heure : "2020-01-01T00:00:00Z"
    # On extrait uniquement l'année et le mois pour l'agrégation mensuelle
    df["validity_time"] = pd.to_datetime(df["validity_time"], errors="coerce")
    df = df.dropna(subset=["validity_time"])
    df["annee_mois"] = df["validity_time"].dt.to_period("M").astype(str)
    df["annee"] = df["validity_time"].dt.year
    df = df[(df["annee"] >= ANNEE_DEBUT) & (df["annee"] <= ANNEE_FIN)]

    # ── ÉTAPE 4 : Conversion des unités ──────────────────────────────────────
    # Température : Kelvin → Celsius
    # 0°C = 273.15 K
    if "t" in df.columns:
        df["t"] = pd.to_numeric(df["t"], errors="coerce")
        df["temp_c"] = df["t"] - 273.15

    # Autres variables : déjà dans les bonnes unités
    for col in ["u", "ff", "fx", "rr1"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # ── ÉTAPE 5 : Agrégation mensuelle par département ────────────────────────
    # Pour chaque département et chaque mois :
    #   - On a plusieurs stations (ex: dept 33 a BORDEAUX + ARCACHON)
    #   - Chaque station a des mesures toutes les 3h (soit ~240 par mois)
    #   - On calcule la moyenne de toutes ces mesures
    # Résultat : 1 ligne par département × mois
    agg_dict = {}
    if "temp_c" in df.columns:
        agg_dict["temp_moy"] = ("temp_c", "mean")
        agg_dict["temp_min"] = ("temp_c", "min")
        agg_dict["temp_max"] = ("temp_c", "max")
    if "u" in df.columns:
        agg_dict["humidite_moy"] = ("u", "mean")
    if "ff" in df.columns:
        agg_dict["vent_moy"] = ("ff", "mean")
    if "fx" in df.columns:
        agg_dict["vent_max"] = ("fx", "max")
    if "rr1" in df.columns:
        agg_dict["precip_total"] = ("rr1", "sum")  # total mensuel en mm (somme des rr1 horaires)

    print(f"\n  Agrégation en cours...")
    df_agg = (
        df.groupby(["dept", "annee_mois"])
        .agg(**agg_dict)
        .reset_index()
    )

    # Arrondir à 2 décimales
    for col in df_agg.select_dtypes(include="float").columns:
        df_agg[col] = df_agg[col].round(2)

    print(
        f"  ✅ dim_meteo : {df_agg.shape[0]:,} lignes × {df_agg.shape[1]} colonnes")
    print(
        f"  Période     : {df_agg['annee_mois'].min()} → {df_agg['annee_mois'].max()}")
    print(f"  Depts       : {df_agg['dept'].nunique()} départements couverts")
    print(f"  Colonnes    : {list(df_agg.columns)}")
    return df_agg


# ══════════════════════════════════════════════════════════════════════════════
# EXÉCUTION
# ══════════════════════════════════════════════════════════════════════════════

# Charger le dictionnaire station → département (créé à l'étape précédente)
df_stations_saved = pd.read_parquet("data/tables/stations_dept.parquet")
station_to_dept = dict(zip(df_stations_saved["ID"], df_stations_saved["dept"]))
print(f"✅ {len(station_to_dept)} stations chargées\n")

# Construire dim_meteo
print("Construction de dim_meteo...")
dim_meteo = build_dim_meteo(station_to_dept)

if not dim_meteo.empty:
    dim_meteo.to_parquet(TABLES_DIR / "dim_meteo.parquet", index=False)
    print(f"\n✅ Sauvegardé → data/tables/dim_meteo.parquet")
    display(dim_meteo.head(10))

✅ 42 stations chargées

Construction de dim_meteo...
  Chargement synop_2020... 168,434 lignes
  Chargement synop_2021... 166,227 lignes
  Chargement synop_2022... 167,883 lignes
  Chargement synop_2023... 170,019 lignes
  Chargement synop_2024... 170,871 lignes
  Chargement synop_2025... 297,699 lignes

  Total brut : 1,141,133 lignes
  Après filtre stations FR : 1,141,133 → 700,363 lignes


/var/folders/ns/x6_m8h8s6pjg7n9j529g49cc0000gn/T/ipykernel_45392/842490905.py:103: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df["annee_mois"] = df["validity_time"].dt.to_period("M").astype(str)



  Agrégation en cours...
  ✅ dim_meteo : 2,883 lignes × 8 colonnes
  Période     : 2020-01 → 2025-12
  Depts       : 41 départements couverts
  Colonnes    : ['dept', 'annee_mois', 'temp_moy', 'temp_min', 'temp_max', 'humidite_moy', 'vent_moy', 'precip_total']

✅ Sauvegardé → data/tables/dim_meteo.parquet


,dept,annee_mois,temp_moy,temp_min,temp_max,humidite_moy,vent_moy,precip_total
0,05,2020-01,3.24,-5.6,14.9,59.93,2.61,24.2
1,05,2020-02,5.20,-3.7,20.9,59.54,2.31,8.4
2,05,2020-03,7.15,-5.4,21.3,58.36,2.42,28.0
3,05,2020-04,12.44,1.7,22.8,49.23,2.88,5.8
4,05,2020-05,15.48,6.1,26.6,65.29,2.25,36.3
5,05,2020-06,16.65,6.0,29.5,65.65,2.10,41.8
6,05,2020-07,21.14,10.3,34.6,55.70,2.38,10.4
7,05,2020-08,20.81,6.8,35.2,54.10,2.43,48.9
8,05,2020-09,16.29,0.4,28.6,65.39,1.95,5.2
9,05,2020-10,9.17,-0.8,21.8,73.13,1.56,40.8


In [23]:
df = pd.read_csv(
    "data/raw/synop/synop_2020.csv.gz",
    sep=";", compression="gzip",
    usecols=["geo_id_wmo", "validity_time"],
    nrows=5000  # 取更多行
)

# Filtrer une seule station pour voir l'intervalle de temps : check les mesures de toutes les 3h pour la station 7005 (Bordeaux) sur les 5000 premières lignes
station = df[df["geo_id_wmo"] == 7005].head(10)
print(station[["geo_id_wmo", "validity_time"]].to_string()) 

     geo_id_wmo         validity_time
0          7005  2020-01-01T00:00:00Z
101        7005  2020-01-01T03:00:00Z
163        7005  2020-01-01T06:00:00Z
200        7005  2020-01-01T09:00:00Z
270        7005  2020-01-01T12:00:00Z
342        7005  2020-01-01T15:00:00Z
402        7005  2020-01-01T18:00:00Z
454        7005  2020-01-01T21:00:00Z
460        7005  2020-01-02T00:00:00Z
527        7005  2020-01-02T03:00:00Z


---
## 4. 🗺️ `dim_geo_pop` — Dimension géographie & population

Fusionne INSEE démographie + INSEE urbanisation + Ameli accès aux soins.

### Fichiers à télécharger → `data/raw/geo_pop/`
| Fichier | URL | Format |
|---|---|---|
| `insee_demographie.csv` | https://www.data.gouv.fr/datasets/pathologies-effectif-de-patients-par-pathologie-sexe-classe-dage-et-territoire-departement-region | CSV ; |
| `insee_urbanisation.csv` | https://www.insee.fr/fr/statistiques/5039853 | CSV ; |
| `ameli_acces_soins.csv` | https://www.assurance-maladie.ameli.fr/etudes-et-donnees/densite-professionnels-sante-liberaux-departement | CSV ; |

In [24]:
# Check les noms des variables dans les bases de données INSEE et Ameli

# INSEE démographie
df = pd.read_csv("data/raw/geo_pop/insee_demographie.csv", sep=";", nrows=3)
print("=== INSEE démographie ===")
print(df.columns.tolist())
display(df.head(2))

# INSEE urbanisation
df2 = pd.read_excel(
    "data/raw/geo_pop/insee_urbanisation.xlsx",
    sheet_name=1,
    header=None,
    skiprows=2  # sauter les 2 premières lignes de métadonnées
)
print("\n=== INSEE urbanisation ===")
display(df2.head(10))

# Ameli (une année) : plusieurs sheets 
df3 = pd.read_excel("data/raw/geo_pop/ameli_acces_soins_2022.xls", nrows=3)
print("\n=== Ameli 2022 ===")
print(df3.columns.tolist())
display(df3.head(2)) 

=== INSEE démographie ===
['annee', 'patho_niv1', 'patho_niv2', 'patho_niv3', 'top', 'cla_age_5', 'sexe', 'region', 'dept', 'Ntop', 'Npop', 'prev', 'Niveau prioritaire', 'libelle_classe_age', 'libelle_sexe', 'tri']


,annee,patho_niv1,patho_niv2,patho_niv3,top,cla_age_5,sexe,region,dept,Ntop,Npop,prev,Niveau prioritaire,libelle_classe_age,libelle_sexe,tri
0,2015,Maladies du foie ou du pancréas (hors mucovisc...,Maladies du foie ou du pancréas (hors mucovisc...,Maladies du foie ou du pancréas (hors mucovisc...,MFP_CAT_EXC,00-04,9,44,57,30,49550,0.054,"1,2,3",de 0 à 4 ans,tous sexes,6.0
1,2015,Maladies du foie ou du pancréas (hors mucovisc...,Maladies du foie ou du pancréas (hors mucovisc...,Maladies du foie ou du pancréas (hors mucovisc...,MFP_CAT_EXC,00-04,9,52,44,30,81200,0.038,"1,2,3",de 0 à 4 ans,tous sexes,6.0



=== INSEE urbanisation ===


,0,1,2
0,01,Ain,67.0
1,02,Aisne,53.2
2,03,Allier,58.3
3,04,Alpes-de-Haute-Provence,61.9
4,05,Hautes-Alpes,59.5
5,06,Alpes-Maritimes,95.9
6,07,Ardèche,63.3
7,08,Ardennes,57.3
8,09,Ariège,55.5
9,10,Aube,61.3



=== Ameli 2022 ===
['Année : ', 2022]


,Année :,2022
0,NaN,NaN
1,Sources :,les données de ce fichier sont issues de Amos ...


In [25]:
def build_dim_geo_pop() -> pd.DataFrame:
    """
    Construit la table dim_geo_pop en fusionnant 3 sources :
      1. INSEE démographie  → population totale, structure par âge
      2. INSEE urbanisation → taux d'urbanisation par département
      3. Ameli (2020-2024)  → densité de professionnels de santé

    CLÉ PRIMAIRE : dept (une ligne par département)

    COLONNES PRODUITES :
    ────────────────────
    dept              → code département (ex: "75")
    pop_totale        → population totale
    part_seniors      → part des 65 ans et plus
    part_jeunes       → part des moins de 15 ans
    tx_urbain         → taux de population en zone urbaine (%)
    densite_med_gen   → densité médecins généralistes /100k hab
    densite_spe       → densité médecins spécialistes /100k hab
    """

    geo_pop_dir = RAW_DIR / "geo_pop"

    # ══════════════════════════════════════════════════════════════════
    # SOURCE 1 : INSEE démographie
    # ══════════════════════════════════════════════════════════════════
    # Ce fichier contient les effectifs de patients par pathologie,
    # âge et département. On l'utilise pour extraire la structure
    # démographique (population par tranche d'âge).
    #
    # Colonnes utiles :
    #   annee      → année
    #   dept       → code département
    #   cla_age_5  → classe d'âge (00-04, 05-09, ..., 85+)
    #   Npop       → population de référence pour ce groupe
    #   sexe       → 9 = tous sexes confondus

    print("Chargement INSEE démographie...")
    df_demo = pd.read_csv(
        geo_pop_dir / "insee_demographie.csv",
        sep=";", encoding="utf-8", low_memory=False
    )
    print(f"  Brut : {len(df_demo):,} lignes")

    # Garder uniquement : tous sexes (sexe=9) + dernière année dispo
    df_demo = df_demo[df_demo["sexe"] == 9].copy()
    derniere_annee = df_demo["annee"].max()
    df_demo = df_demo[df_demo["annee"] == derniere_annee]
    print(
        f"  Après filtre (sexe=9, annee={derniere_annee}) : {len(df_demo):,} lignes")

    # Nettoyage département
    df_demo["dept"] = (
        df_demo["dept"].astype(str).str.strip().str.upper().str.zfill(2)
    )

    # Conversion numérique
    df_demo["Npop"] = pd.to_numeric(df_demo["Npop"], errors="coerce")
    df_demo["cla_age_5"] = df_demo["cla_age_5"].astype(str).str.strip()
    
    # Problème : Le fichier contient une ligne par combinaison pathologie × tranche d'âge × département, ce qui entraîne un comptage multiple de Npop lors de l'agrégation.
    # Solution : On filtre sur patho_niv1 == "Total consommants tous régimes", qui représente la population totale par tranche d'âge sans doublon pathologique.

    # Garder uniquement "Total consommants tous régimes"
    # → évite le double-comptage par pathologie
    df_demo = df_demo[
        df_demo["patho_niv1"] == "Total consommants tous régimes"
    ].copy()
    print(f"  Après filtre patho totale : {len(df_demo):,} lignes")

    # Population totale par département
    df_pop_totale = (
        df_demo.groupby("dept")["Npop"]
        .sum()
        .reset_index()
        .rename(columns={"Npop": "pop_totale"})
        )

    # Seniors : 65 ans et plus
    df_seniors = (
        df_demo[df_demo["cla_age_5"].str[:2].isin(
            ["65", "70", "75", "80", "85", "90", "95"]
        )]
        .groupby("dept")["Npop"].sum()
        .reset_index()
        .rename(columns={"Npop": "nb_seniors"})
    )

    # Jeunes : moins de 15 ans
    df_jeunes = (
        df_demo[df_demo["cla_age_5"].str[:2].isin(["00", "05", "10"])]
        .groupby("dept")["Npop"].sum()
        .reset_index()
        .rename(columns={"Npop": "nb_jeunes"})
    )

    # Fusion et calcul des parts
    df_demographie = df_pop_totale.copy()
    df_demographie = df_demographie.merge(df_seniors, on="dept", how="left")
    df_demographie = df_demographie.merge(df_jeunes,  on="dept", how="left")
    df_demographie["part_seniors"] = (
        df_demographie["nb_seniors"] / df_demographie["pop_totale"]
    ).round(4)
    df_demographie["part_jeunes"] = (
        df_demographie["nb_jeunes"] / df_demographie["pop_totale"]
    ).round(4)
    df_demographie = df_demographie[["dept", "pop_totale",
                                     "part_seniors", "part_jeunes"]]

    print(f"  ✅ Démographie : {len(df_demographie)} départements")

    # ══════════════════════════════════════════════════════════════════
    # SOURCE 2 : INSEE urbanisation
    # ══════════════════════════════════════════════════════════════════
    # Le fichier Excel a une structure complexe avec des lignes de
    # métadonnées en haut. On cherche la ligne contenant les codes
    # département pour trouver le bon skiprows.

    print("\nChargement INSEE urbanisation...")
    df_urban = pd.read_excel(
        geo_pop_dir / "insee_urbanisation.xlsx",
        sheet_name=1,      # 2ème feuille
        header=None,
        skiprows=2,        # sauter les 2 lignes de métadonnées
        usecols=[0, 2]     # colonne 0=dept, colonne 2=taux
        )
    
    df_urban.columns = ["dept", "tx_urbain"]
    df_urban["dept"] = (df_urban["dept"].astype(str).str.strip().str.zfill(2).str.upper())
    df_urban["tx_urbain"] = pd.to_numeric(df_urban["tx_urbain"], errors="coerce").round(1)
    df_urban = df_urban[df_urban["dept"].isin(DEPTS)].dropna()
    print(f"  ✅ Urbanisation : {len(df_urban)} départements")

    # ══════════════════════════════════════════════════════════════════
    # SOURCE 3 : Ameli — densité professionnels de santé
    # ══════════════════════════════════════════════════════════════════
    # 5 fichiers annuels (2020-2024), chacun avec plusieurs feuilles.
    # On prend la feuille "Généralistes et MEP" pour les médecins
    # généralistes et "Spécialistes" pour les spécialistes.
    #
    # Structure de chaque feuille :
    #   col0 → type de spécialité (ex: "01- Médecine générale")
    #   col1 → département (ex: "01- Ain")
    #   col2 → effectif
    #   col3 → population
    #   col4 → densité /100 000 hab  ← on prend ça

    print("\nChargement Ameli...")

    def charger_ameli_feuille(fpath: Path, feuille: str,
                              filtre_type: str) -> pd.DataFrame:
        """
        Charge une feuille d'un fichier Ameli et retourne la densité
        pour un type de professionnel donné, par département.
        """
        df = pd.read_excel(fpath, sheet_name=feuille, header=0)
        df.columns = ["type_ps", "dept_raw", "effectif",
                      "population", "densite"] + list(df.columns[5:])

        # Filtrer sur le type de professionnel voulu
        df = df[df["type_ps"].astype(str).str.contains(
            filtre_type, case=False, na=False
        )].copy()

        # Extraire le code département (2 premiers chiffres)
        df["dept"] = (
            df["dept_raw"].astype(str)
            .str.extract(r"^(\d{2}|2[AB])", expand=False)
            .str.zfill(2)
        )
        df["densite"] = pd.to_numeric(df["densite"], errors="coerce")
        df = df[df["dept"].isin(DEPTS)]

        return df.groupby("dept")["densite"].mean().reset_index()

    # Charger les 5 années et faire la moyenne
    annees_ameli = range(2020, 2025)
    dfs_gen = []
    dfs_spe = []

    for annee in annees_ameli:
        fpath = geo_pop_dir / f"ameli_acces_soins_{annee}.xls"
        if not fpath.exists():
            print(f"  ⚠️  Manquant : {fpath.name}")
            continue
        try:
            df_gen = charger_ameli_feuille(
                fpath, "Généralistes et MEP", "Médecine générale"
            )
            df_gen["annee"] = annee
            dfs_gen.append(df_gen)

            df_spe = charger_ameli_feuille(
                fpath, "Spécialistes", "Médecine"
            )
            df_spe["annee"] = annee
            dfs_spe.append(df_spe)
            print(f"  ✅ Ameli {annee} chargé")
        except Exception as e:
            print(f"  ⚠️  Erreur {annee} : {e}")

    # Moyenne sur toutes les années disponibles
    if dfs_gen:
        df_ameli_gen = (
            pd.concat(dfs_gen)
            .groupby("dept")["densite"]
            .mean()
            .round(2)
            .reset_index()
            .rename(columns={"densite": "densite_med_gen"})
        )
    else:
        df_ameli_gen = pd.DataFrame(columns=["dept", "densite_med_gen"])

    if dfs_spe:
        df_ameli_spe = (
            pd.concat(dfs_spe)
            .groupby("dept")["densite"]
            .mean()
            .round(2)
            .reset_index()
            .rename(columns={"densite": "densite_spe"})
        )
    else:
        df_ameli_spe = pd.DataFrame(columns=["dept", "densite_spe"])

    print(f"  ✅ Ameli généralistes : {len(df_ameli_gen)} depts")
    print(f"  ✅ Ameli spécialistes : {len(df_ameli_spe)} depts")

    # ══════════════════════════════════════════════════════════════════
    # FUSION FINALE
    # ══════════════════════════════════════════════════════════════════
    # On part de la liste complète des départements et on joint
    # chaque source avec how="left" pour conserver tous les depts
    # même si une source est incomplète

    print("\nFusion des 3 sources...")
    dim_geo_pop = pd.DataFrame({"dept": DEPTS})
    dim_geo_pop = dim_geo_pop.merge(df_demographie, on="dept", how="left")
    dim_geo_pop = dim_geo_pop.merge(df_urban,       on="dept", how="left")
    dim_geo_pop = dim_geo_pop.merge(df_ameli_gen,   on="dept", how="left")
    dim_geo_pop = dim_geo_pop.merge(df_ameli_spe,   on="dept", how="left")

    # Rapport de couverture
    print("\n📊 Couverture :")
    for col in dim_geo_pop.columns[1:]:
        pct = dim_geo_pop[col].notna().mean() * 100
        barre = "█" * int(pct // 10) + "░" * (10 - int(pct // 10))
        statut = "✅" if pct > 80 else "⚠️ "
        print(f"  {statut} {col:<25} {barre} {pct:.0f}%")

    print(f"\n✅ dim_geo_pop : {dim_geo_pop.shape[0]} depts × "
          f"{dim_geo_pop.shape[1]} colonnes")
    return dim_geo_pop


# ══════════════════════════════════════════════════════════════════════════════
# EXÉCUTION
# ══════════════════════════════════════════════════════════════════════════════
dim_geo_pop = build_dim_geo_pop()

if not dim_geo_pop.empty:
    dim_geo_pop.to_parquet(TABLES_DIR / "dim_geo_pop.parquet", index=False)
    print(f"\n✅ Sauvegardé → data/tables/dim_geo_pop.parquet")
    display(dim_geo_pop.head(10))

Chargement INSEE démographie...
  Brut : 5,216,400 lignes
  Après filtre (sexe=9, annee=2023) : 199,080 lignes
  Après filtre patho totale : 2,520 lignes
  ✅ Démographie : 102 départements

Chargement INSEE urbanisation...
  ✅ Urbanisation : 96 départements

Chargement Ameli...
  ✅ Ameli 2020 chargé
  ✅ Ameli 2021 chargé
  ✅ Ameli 2022 chargé
  ✅ Ameli 2023 chargé
  ✅ Ameli 2024 chargé
  ✅ Ameli généralistes : 96 depts
  ✅ Ameli spécialistes : 96 depts

Fusion des 3 sources...

📊 Couverture :
  ✅ pop_totale                ██████████ 100%
  ✅ part_seniors              ██████████ 100%
  ✅ part_jeunes               ██████████ 100%
  ✅ tx_urbain                 ██████████ 100%
  ✅ densite_med_gen           ██████████ 100%
  ✅ densite_spe               ██████████ 100%

✅ dim_geo_pop : 96 depts × 7 colonnes

✅ Sauvegardé → data/tables/dim_geo_pop.parquet


,dept,pop_totale,part_seniors,part_jeunes,tx_urbain,densite_med_gen,densite_spe
0,01,1264530,0.1007,0.0914,67.0,5.88,0.11
1,02,1025570,0.1137,0.0856,53.2,6.48,0.16
2,03,653420,0.1488,0.0693,58.3,7.69,0.34
3,04,333290,0.1394,0.0711,61.9,9.92,0.43
4,05,290280,0.1340,0.0714,59.5,12.47,0.61
5,06,2314380,0.1253,0.0744,95.9,10.82,0.59
6,07,661550,0.1334,0.0738,63.3,7.25,0.43
7,08,515650,0.1205,0.0802,57.3,7.87,0.16
8,09,307080,0.1413,0.0684,55.5,8.73,0.28
9,10,593350,0.1217,0.0817,61.3,6.25,0.23


---
## 5. 💼 `dim_csp` — Catégories Socio-Professionnelles

**Source :** INSEE  
**Fichier :** `data/raw/insee_csp.csv`  
**URL :** https://www.insee.fr/fr/statistiques/2012721#tableau-TCRD_014_tab1_departements

In [26]:
def build_dim_csp() -> pd.DataFrame:
    """
    Construit la table dim_csp à partir du fichier INSEE CSP (feuille "DEP").

    CLÉ PRIMAIRE : dept
    COLONNES : tx_agriculteurs, tx_artisans, tx_cadres, tx_prof_interm,
               tx_employes, tx_ouvriers, tx_autres  (en décimales, ex: 0.12)
    """
    fpath = RAW_DIR / "insee_csp" / "insee_csp.xlsx"
    if not fpath.exists():
        print(f"⚠️  Fichier manquant : {fpath}")
        return pd.DataFrame()

    df = pd.read_excel(fpath, sheet_name="DEP", header=None, skiprows=4)
    df.columns = ["dept", "nom", "tx_agriculteurs", "tx_artisans",
                  "tx_cadres", "tx_prof_interm", "tx_employes",
                  "tx_ouvriers", "tx_autres"]

    df["dept"] = df["dept"].astype(str).str.strip().str.zfill(2).str.upper()
    df = df[df["dept"].isin(DEPTS)]

    for col in df.columns[2:]:
        df[col] = pd.to_numeric(df[col], errors="coerce") / 100

    cols_out = ["dept"] + list(df.columns[2:])
    df = df[cols_out].reset_index(drop=True)

    print(f"✅ dim_csp : {len(df)} départements × {df.shape[1]} colonnes")
    return df


dim_csp = build_dim_csp()

if not dim_csp.empty:
    dim_csp.to_parquet(TABLES_DIR / "dim_csp.parquet", index=False)
    print(f"✅ Sauvegardé → data/tables/dim_csp.parquet")
    display(dim_csp.head(5))


✅ dim_csp : 96 départements × 8 colonnes
✅ Sauvegardé → data/tables/dim_csp.parquet


,dept,tx_agriculteurs,tx_artisans,tx_cadres,tx_prof_interm,tx_employes,tx_ouvriers,tx_autres
0,01,0.009,0.065,0.163,0.271,0.253,0.232,0.007
1,02,0.017,0.049,0.087,0.228,0.293,0.303,0.025
2,03,0.030,0.065,0.096,0.229,0.305,0.262,0.013
3,04,0.027,0.101,0.124,0.258,0.278,0.203,0.010
4,05,0.027,0.096,0.107,0.282,0.299,0.183,0.006


---
## 6A. 💨 `dim_qualite_air` — Qualité de l'air (AASQA)

**Source :** data.gouv.fr — Indices de qualité de l'air (AASQA)  
**URL :** https://www.data.gouv.fr/datasets/donnees-temps-reel-de-mesure-des-concentrations-de-polluants-atmospheriques-reglementes-1

### Fichier à télécharger → `data/raw/AASQA/`
Sur la page data.gouv.fr, téléchargez les fichiers CSV annuels (2021 → 2025), pas d'info en 2020.  
Format attendu : colonnes `date`, `code_zone` (code dept), `valeur`, `qualificatif`

> **Pourquoi c'est critique :** O₃ et PM2.5 déclenchent directement les crises d'asthme. NO₂ potentialise les réponses allergiques. La pollution particulaire fragilise les voies respiratoires des nourrissons (bronchiolite).

In [27]:
df = pd.read_csv(
    "data/raw/AASQA/FR_E2_2021-01-01.csv",
    sep=";", nrows=5
)
print(df.columns.tolist())
display(df.head())

['Date de début', 'Date de fin', 'Organisme', 'code zas', 'Zas', 'code site', 'nom site', "type d'implantation", 'Polluant', "type d'influence", 'discriminant', 'Réglementaire', "type d'évaluation", 'procédure de mesure', 'type de valeur', 'valeur', 'valeur brute', 'unité de mesure', 'taux de saisie', 'couverture temporelle', 'couverture de données', 'code qualité', 'validité']


,Date de début,Date de fin,Organisme,code zas,Zas,code site,nom site,type d'implantation,Polluant,type d'influence,...,procédure de mesure,type de valeur,valeur,valeur brute,unité de mesure,taux de saisie,couverture temporelle,couverture de données,code qualité,validité
0,2021/01/01 00:00:00,2021/01/01 01:00:00,ATMO GRAND EST,FR44ZAG02,ZAG METZ,FR01005,Hayange,Périurbaine,PM10,Industrielle,...,Auto PM_Conf_app MP101M-RST,moyenne horaire validée,18.9,18.875,µg-m3,NaN,NaN,NaN,A,1
1,2021/01/01 01:00:00,2021/01/01 02:00:00,ATMO GRAND EST,FR44ZAG02,ZAG METZ,FR01005,Hayange,Périurbaine,PM10,Industrielle,...,Auto PM_Conf_app MP101M-RST,moyenne horaire validée,10.8,10.800,µg-m3,NaN,NaN,NaN,A,1
2,2021/01/01 02:00:00,2021/01/01 03:00:00,ATMO GRAND EST,FR44ZAG02,ZAG METZ,FR01005,Hayange,Périurbaine,PM10,Industrielle,...,Auto PM_Conf_app MP101M-RST,moyenne horaire validée,10.3,10.300,µg-m3,NaN,NaN,NaN,A,1
3,2021/01/01 03:00:00,2021/01/01 04:00:00,ATMO GRAND EST,FR44ZAG02,ZAG METZ,FR01005,Hayange,Périurbaine,PM10,Industrielle,...,Auto PM_Conf_app MP101M-RST,moyenne horaire validée,6.4,6.400,µg-m3,NaN,NaN,NaN,A,1
4,2021/01/01 04:00:00,2021/01/01 05:00:00,ATMO GRAND EST,FR44ZAG02,ZAG METZ,FR01005,Hayange,Périurbaine,PM10,Industrielle,...,Auto PM_Conf_app MP101M-RST,moyenne horaire validée,8.1,8.050,µg-m3,NaN,NaN,NaN,A,1


In [28]:
import re

def _extract_dept_from_site(code) -> str | None:
    """Extrait le code département depuis un code site AASQA (ex: FR01005 → '01')."""
    if pd.isna(code):
        return None
    code = str(code).strip()
    if not code.startswith("FR"):
        return None
    reste = code[2:]
    if reste.startswith("2A") or reste.startswith("2B"):
        return reste[:2]
    if len(reste) >= 2 and reste[:2].isdigit():
        return reste[:2].zfill(2)
    return None


def build_dim_qualite_air() -> pd.DataFrame:
    """
    Construit la table dim_qualite_air à partir des 60 fichiers AASQA mensuels.

    SOURCE : LCSQA — Concentrations de polluants atmosphériques réglementés
    Fichiers : FR_E2_YYYY-MM-01.csv (un par mois, 2021-2025)

    EXTRACTION DU DÉPARTEMENT :
    ───────────────────────────
    Le code site suit le format : FR + 2 chiffres dept + 3 chiffres station
    Ex : FR01005 → département 01 (Ain)
         FR75001 → département 75 (Paris)
         FR2A001 → département 2A (Corse-du-Sud)

    POLLUANTS TRAITÉS :
    ───────────────────
    PM10  → particules fines (µg/m³) — facteur asthme & bronchiolite
    PM2.5 → particules très fines     — facteur asthme sévère
    NO2   → dioxyde d'azote           — facteur allergie + asthme
    O3    → ozone                     — déclencheur crises d'asthme

    COLONNES PRODUITES :
    ────────────────────
    dept          → code département
    annee_mois    → période (ex: "2021-03")
    pm10_moy      → concentration mensuelle moyenne PM10 (µg/m³)
    pm25_moy      → concentration mensuelle moyenne PM2.5
    no2_moy       → concentration mensuelle moyenne NO2
    o3_moy        → concentration mensuelle moyenne O3
    nb_jours_pm10_eleve → nb jours avec PM10 > 50 µg/m³ (seuil OMS)

    CLÉ PRIMAIRE : dept × annee_mois
    """

    aasqa_dir = RAW_DIR / "AASQA"
    if not aasqa_dir.exists():
        print(f"⚠️  Dossier manquant : {aasqa_dir}")
        return pd.DataFrame()

    fichiers = sorted(aasqa_dir.glob("FR_E2_*.csv"))
    if not fichiers:
        print("⚠️  Aucun fichier FR_E2_*.csv trouvé")
        return pd.DataFrame()

    print(f"Chargement de {len(fichiers)} fichiers AASQA...")

    # Polluants d'intérêt et leurs noms normalisés
    POLLUANTS = {
        "PM10":  "pm10",
        "PM2.5": "pm25",
        "NO2":   "no2",
        "O3":    "o3",
        "NO":    "no",    # optionnel
        "SO2":   "so2",   # optionnel
    }

    dfs = []

    for fpath in fichiers:

        # Extraire l'année et le mois depuis le nom de fichier
        # Format : FR_E2_2021-01-01.csv → annee_mois = "2021-01"
        match = re.search(r'FR_E2_(\d{4}-\d{2})-\d{2}\.csv', fpath.name)
        if not match:
            continue
        annee_mois = match.group(1)

        try:
            df = pd.read_csv(
                fpath,
                sep=";",
                encoding="utf-8",
                low_memory=False,
                dtype=str          # tout en texte d'abord
            )

            # ── Colonnes utiles ──────────────────────────────────────────────
            # Renommer pour standardiser
            rename = {
                "Date de début":    "date_debut",
                "code site":        "code_site",
                "Polluant":         "polluant",
                "valeur":           "valeur",
                "validité":         "validite",
                "unité de mesure":  "unite",
            }
            df = df.rename(columns={k: v for k, v in rename.items()
                                    if k in df.columns})

                df["dept"] = df["code_site"].apply(_extract_dept_from_site)
            df = df.dropna(subset=["dept"])
            df = df[df["dept"].isin(DEPTS)]

            # ── Filtrage sur les polluants d'intérêt ─────────────────────
            df = df[df["polluant"].isin(POLLUANTS.keys())].copy()
            if df.empty:
                continue

            # ── Filtrage sur données valides (validite = 1) ───────────────
            df["validite"] = pd.to_numeric(df["validite"], errors="coerce")
            df = df[df["validite"] == 1]

            # ── Conversion numérique de la valeur ─────────────────────────
            df["valeur"] = pd.to_numeric(df["valeur"], errors="coerce")
            df = df.dropna(subset=["valeur"])

            # ── Nettoyage des valeurs aberrantes ──────────────────────────
            # Valeurs négatives ou extrêmes → NaN
            df.loc[df["valeur"] < 0, "valeur"] = np.nan
            df.loc[df["valeur"] > 1000, "valeur"] = np.nan

            df["annee_mois"] = annee_mois

            # ── Normaliser le nom du polluant ──────────────────────────────
            df["polluant_norm"] = df["polluant"].map(POLLUANTS)

            dfs.append(df[["dept", "annee_mois", "polluant_norm",
                            "valeur"]].copy())

        except Exception as e:
            print(f"  ⚠️  Erreur {fpath.name} : {e}")

    if not dfs:
        print("❌ Aucune donnée chargée")
        return pd.DataFrame()

    df_all = pd.concat(dfs, ignore_index=True)
    print(f"Total lignes valides : {len(df_all):,}")
    print(f"Polluants disponibles : {df_all['polluant_norm'].unique().tolist()}")

    # ── Agrégation mensuelle par département × polluant ───────────────────
    # Chaque fichier contient déjà un seul mois → groupby dept × polluant
    df_pivot = (
        df_all.groupby(["dept", "annee_mois", "polluant_norm"])["valeur"]
        .mean()
        .round(2)
        .reset_index()
    )

    # Pivot : une colonne par polluant
    df_wide = df_pivot.pivot_table(
        index=["dept", "annee_mois"],
        columns="polluant_norm",
        values="valeur",
        aggfunc="mean"
    ).reset_index()

    # Renommer les colonnes
    df_wide.columns.name = None
    rename_cols = {p: f"{p}_moy" for p in POLLUANTS.values()
                   if p in df_wide.columns}
    df_wide = df_wide.rename(columns=rename_cols)

    # ── Indicateur jours PM10 élevé ──────────────────────────────────────
    # Compte le nombre de jours calendaires distincts où PM10 > 50 µg/m³
    if "pm10" in df_all["polluant_norm"].values:
        df_pm10 = df_all[df_all["polluant_norm"] == "pm10"].copy()
        # Reconstruire la date réelle depuis date_debut si disponible,
        # sinon on sait que chaque fichier = 1 mois → on approche par mesure unique
        df_pm10["jour_eleve"] = (df_pm10["valeur"] > 50).astype(int)
        # Moyenne par (dept, annee_mois) : proportion de mesures dépassant le seuil
        # (proxy du nb de jours, car les mesures sont ~horaires)
        df_jours = (
            df_pm10.groupby(["dept", "annee_mois"])["jour_eleve"]
            .mean()
            .mul(30)         # × 30 jours → estimation du nb de jours/mois
            .round(1)
            .reset_index()
            .rename(columns={"jour_eleve": "nb_jours_pm10_eleve_est"})
        )
        df_wide = df_wide.merge(df_jours, on=["dept", "annee_mois"],
                                how="left")

    print(f"\n✅ dim_qualite_air : {df_wide.shape[0]:,} lignes × "
          f"{df_wide.shape[1]} colonnes")
    print(f"Période      : {df_wide['annee_mois'].min()} → "
          f"{df_wide['annee_mois'].max()}")
    print(f"Départements : {df_wide['dept'].nunique()} couverts")
    print(f"Colonnes     : {list(df_wide.columns)}")

    # Couverture par polluant
    print("\n📊 Couverture par polluant :")
    for col in [c for c in df_wide.columns if c.endswith("_moy")]:
        pct   = df_wide[col].notna().mean() * 100
        barre = "█" * int(pct // 10) + "░" * (10 - int(pct // 10))
        print(f"  {col:<20} {barre} {pct:.0f}%")

    return df_wide


# ══════════════════════════════════════════════════════════════════════════════
# EXÉCUTION
# ══════════════════════════════════════════════════════════════════════════════
print("Construction de dim_qualite_air...")
dim_qualite_air = build_dim_qualite_air()

if not dim_qualite_air.empty:
    dim_qualite_air.to_parquet(
        TABLES_DIR / "dim_qualite_air.parquet", index=False
    )
    print(f"\n✅ Sauvegardé → data/tables/dim_qualite_air.parquet")
    display(dim_qualite_air.head(10))

IndentationError: unexpected indent (1821968191.py, line 107)

---
## 6B. 🌿 `dim_pollen` — Concentration pollinique (RNSA)

**Source :** Réseau National de Surveillance Aérobiologique (RNSA)  
**URL :** https://www.data.gouv.fr/datasets/donnees-historiques-de-surveillance-des-pollens-et-des-moisissures

### Fichier à télécharger → `data/raw/pollen/`
> **Ce dossier rassemble les mesures effectuées sur 133 capteurs entre 1987 et 2024 et compilées par le RNSA. Chaque fichier contient les données d'une année et d'un capteur. Les données sont ici présentées par jour et par taxons.

In [29]:
pollen_dir = Path("data/raw/BDD_daily")

# Filtrer uniquement les fichiers 2020-2025
fichiers_utiles = []
for f in pollen_dir.glob("*.xls"):
    match = re.search(r'(\d{4})-\d{2}-\d{2}_to_', f.name)
    if match:
        annee = int(match.group(1))
        if 2020 <= annee <= 2025:
            fichiers_utiles.append(f)

print(f"Fichiers 2020-2025 : {len(fichiers_utiles)}")
for f in sorted(fichiers_utiles)[:10]:
    print(f"  {f.name}")
    
import shutil

pollen_dir = Path("data/raw/BDD_daily")
dest_dir = Path("data/raw/BDD_daily_2020_2025")
dest_dir.mkdir(parents=True, exist_ok=True)

copiés = 0
for f in pollen_dir.glob("*.xls"):
    match = re.search(r'(\d{4})-\d{2}-\d{2}_to_', f.name)
    if match:
        annee = int(match.group(1))
        if 2020 <= annee <= 2025:
            shutil.copy2(f, dest_dir / f.name)
            copiés += 1

print(f"✅ {copiés} fichiers copiés → {dest_dir}")

Fichiers 2020-2025 : 253
  Particle_Extract_AGEN_2020-01-01_to_2020-12-31.xls
  Particle_Extract_AGEN_2021-01-01_to_2021-12-31.xls
  Particle_Extract_AGEN_2023-01-01_to_2023-12-31.xls
  Particle_Extract_AIXENPRO_2020-01-01_to_2020-12-31.xls
  Particle_Extract_AIXENPRO_2021-01-01_to_2021-12-31.xls
  Particle_Extract_AIXENPRO_2023-01-01_to_2023-12-31.xls
  Particle_Extract_AJACCIO_2020-01-01_to_2020-12-31.xls
  Particle_Extract_AJACCIO_2021-01-01_to_2021-12-31.xls
  Particle_Extract_AJACCIO_2023-01-01_to_2023-12-31.xls
  Particle_Extract_AMBERIEU_2020-01-01_to_2020-12-31.xls
✅ 253 fichiers copiés → data/raw/BDD_daily_2020_2025


In [30]:
pollen_dir = Path("data/raw/BDD_daily_2020_2025")
villes = set()
for f in pollen_dir.glob("*.xls"):
    match = re.search(r'Particle_Extract_(.+?)_\d{4}', f.name)
    if match:
        villes.add(match.group(1))

print(f"Total villes : {len(villes)}")
for v in sorted(villes):
    print(f"  {v}")

Total villes : 93
  AGEN
  AIXENPRO
  AJACCIO
  AMBERIEU
  AMIENS
  ANDORRA
  ANGERS
  ANGOULEM
  ANNECY
  ANNEMASS
  ANTONY
  AURILLAC
  AVIGNON
  BAGNOLS
  BART
  BERRIAS
  BESANCON
  BLETTERA
  BORDPESS
  BOURGENB
  BOURGES
  BOURGOIN
  BREST
  BRUSSLAN
  CAEN
  CASTRES
  CHALON-S
  CHAMBERY
  CHARLEVI
  CHAUMONT
  CHOLET
  CLERMONT
  DIJON
  DINAN
  DOLE
  DRAGUIGN
  GAP
  GENAS
  GLEIZE
  GONESSE
  GRENOBLE
  LAROCHE-
  LAROCHL
  LEMANS
  LEPUYENV
  LILLE
  LIMOGES
  LORIENT
  LURE
  LYON
  MACON
  MAREUIL
  MARSEILL
  METZ
  MONTLUCO
  MONTPELL
  MTMARSAN
  MULHOUSE
  NANCY
  NANTES
  NARBONNE
  NEVERS
  NICE
  NICE2
  NIORT
  ORLEANS
  PARIS
  PERIGUEU
  POITIERS
  PONTIVY
  REIMS
  RENNES
  ROANNE
  ROUEN
  ROUSSILL
  SACLAY
  SACLAYSP
  SAINT-ET
  SAINTDIE
  ST-BRIEU
  STALBAN
  STEFOY
  STRASBOU
  TOULON
  TOULOUSE
  TOULOUSM
  TOURS
  TROYES
  TULLE
  VALDAHON
  VALENCE
  VICHY
  VILLENEU


In [31]:
# ══════════════════════════════════════════════════════════════════════════════
# TABLE DE CORRESPONDANCE : ville RNSA → code département
# ══════════════════════════════════════════════════════════════════════════════
# Construite manuellement à partir des 93 villes du dataset

VILLE_TO_DEPT = {
    "AGEN":       "47",  # Lot-et-Garonne
    "AIXENPRO":   "13",  # Bouches-du-Rhône (Aix-en-Provence)
    "AJACCIO":    "2A",  # Corse-du-Sud
    "AMBERIEU":   "01",  # Ain (Ambérieu-en-Bugey)
    "AMIENS":     "80",  # Somme
    "ANDORRA":    None,  # Andorre → hors France, ignoré
    "ANGERS":     "49",  # Maine-et-Loire
    "ANGOULEM":   "16",  # Charente (Angoulême)
    "ANNECY":     "74",  # Haute-Savoie
    "ANNEMASS":   "74",  # Haute-Savoie (Annemasse)
    "ANTONY":     "92",  # Hauts-de-Seine
    "AURILLAC":   "15",  # Cantal
    "AVIGNON":    "84",  # Vaucluse
    "BAGNOLS":    "30",  # Gard (Bagnols-sur-Cèze)
    "BART":       "25",  # Doubs
    "BERRIAS":    "07",  # Ardèche
    "BESANCON":   "25",  # Doubs
    "BEZIERS":    "34",  # Hérault
    "BORDEAUX":   "33",  # Gironde
    "BOURGES":    "18",  # Cher
    "BREST":      "29",  # Finistère
    "BRIANCON":   "05",  # Hautes-Alpes
    "CAEN":       "14",  # Calvados
    "CARCASS":    "11",  # Aude (Carcassonne)
    "CHAMBERY":   "73",  # Savoie
    "CHARLEV":    "08",  # Ardennes (Charleville-Mézières)
    "CHARTRES":   "28",  # Eure-et-Loir
    "CHERBOU":    "50",  # Manche (Cherbourg)
    "CLERMON":    "63",  # Puy-de-Dôme (Clermont-Ferrand)
    "COLMAR":     "68",  # Haut-Rhin
    "CREIL":      "60",  # Oise
    "DIJON":      "21",  # Côte-d'Or
    "DIGNE":      "04",  # Alpes-de-Haute-Provence
    "DUNKERQ":    "59",  # Nord (Dunkerque)
    "EMBRUN":     "05",  # Hautes-Alpes
    "GRENOBLE":   "38",  # Isère
    "LAON":       "02",  # Aisne
    "LAROCHE":    "89",  # Yonne (Laroche-Saint-Cydroine / Auxerre)
    "LAVAL":      "53",  # Mayenne
    "LEMANS":     "72",  # Sarthe (Le Mans)
    "LENS":       "62",  # Pas-de-Calais
    "LILE":       "59",  # Nord (Lille)
    "LIMOGES":    "87",  # Haute-Vienne
    "LORIENT":    "56",  # Morbihan
    "LYON":       "69",  # Rhône
    "MARSEILL":   "13",  # Bouches-du-Rhône
    "METZ":       "57",  # Moselle
    "MONTLUEL":   "01",  # Ain
    "MONTPELL":   "34",  # Hérault (Montpellier)
    "MULHOUSE":   "68",  # Haut-Rhin
    "NANCY":      "54",  # Meurthe-et-Moselle
    "NANTES":     "44",  # Loire-Atlantique
    "NICE":       "06",  # Alpes-Maritimes
    "NIMES":      "30",  # Gard
    "NIORT":      "79",  # Deux-Sèvres
    "ORLEANS":    "45",  # Loiret
    "PARIS":      "75",  # Paris
    "PAU":        "64",  # Pyrénées-Atlantiques
    "PERPIGNA":   "66",  # Pyrénées-Orientales (Perpignan)
    "POITIERS":   "86",  # Vienne
    "REIMS":      "51",  # Marne
    "RENNES":     "35",  # Ille-et-Vilaine
    "ROUEN":      "76",  # Seine-Maritime
    "SAINTBRI":   "22",  # Côtes-d'Armor (Saint-Brieuc) — corrigé depuis 71
    "SAINTETIE":  "42",  # Loire (Saint-Étienne)
    "SAINTMAL":   "35",  # Ille-et-Vilaine (Saint-Malo)
    "SAINTQUEN":  "80",  # Somme (Saint-Quentin)
    "SARREBOU":   "57",  # Moselle (Sarreboug)
    "STALBAN":    "81",  # Tarn (Saint-Alban)
    "STEFOY":     "33",  # Gironde (Sainte-Foy-la-Grande)
    "STRASBOU":   "67",  # Bas-Rhin (Strasbourg)
    "TOULON":     "83",  # Var
    "TOULOUSE":   "31",  # Haute-Garonne
    "TOULOUSM":   "31",  # Haute-Garonne (autre station Toulouse)
    "TOURS":      "37",  # Indre-et-Loire
    "TROYES":     "10",  # Aube
    "VALENCE":    "26",  # Drôme
    "VANNES":     "56",  # Morbihan
    "VERSAIL":    "78",  # Yvelines (Versailles)
    "VICHY":      "03",  # Allier
    "VIENNE":     "38",  # Isère (Vienne)
    "VILLENEU":   "47",  # Lot-et-Garonne (Villeneuve-sur-Lot)
    "ST-BRIEU":   "22",  # Côtes-d'Armor (Saint-Brieuc)
    "BLETTERA":   "07",  # Ardèche (Bletterans → 39 Jura ?)
    "MONTBRIS":   "42",  # Loire (Montbrison)
    "AUXERRE":    "89",  # Yonne
    "CHALONS":    "51",  # Marne (Châlons-en-Champagne)
    "CHARLEV2":   "08",  # Ardennes
    "BELFORT":    "90",  # Territoire de Belfort
    "MONTELIM":   "26",  # Drôme (Montélimar)
    "RODEZ":      "12",  # Aveyron
    "TARBES":     "65",  # Hautes-Pyrénées
    "BASTIA":     "2B",  # Haute-Corse
    "TOUQUET":    "62",  # Pas-de-Calais (Le Touquet)
    "BIARRIT":    "64",  # Pyrénées-Atlantiques (Biarritz)
    "CAHORS":     "46",  # Lot
    "PERIGUEU":   "24",  # Dordogne (Périgueux)
    "EVREUX":     "27",  # Eure
    "ABBEVIL":    "80",  # Somme (Abbeville)
    "ALENCON":    "61",  # Orne
    "CHAUMONT":   "52",  # Haute-Marne
    "COLOGNAC":   "30",  # Gard
}

# Pollen clés pour les pathologies respiratoires
POLLEN_CLES = {
    "AMBROSIA":  "ambrosia",   # Ambroisie  → allergie sévère (août-sept)
    "ALNUS":     "alnus",      # Aulne/Aulnaie → allergie hiver-printemps
    "BETULA":    "betula",     # Bouleau    → allergie printemps
    "ARTEMISI":  "artemisia",  # Armoise    → allergie été
    "GRAMINE":   "graminees",  # Graminées  → allergie été (le + important)
    "GRAMINEES": "graminees",  # variante du nom
    "POACEAE":   "graminees",  # variante scientifique
    "CUPRESSU":  "cypres",     # Cyprès     → allergie hiver (Sud)
    "PLATANUS":  "platane",    # Platane    → allergie printemps (villes)
    "URTICA":    "urticacees", # Urticacées → allergie été
}


def build_dim_pollen() -> pd.DataFrame:
    """
    Construit la table dim_pollen à partir des 253 fichiers RNSA BDD_daily.

    STRATÉGIE :
    ───────────
    1. Charger chaque fichier (ville × année)
    2. Rattacher la ville à un département via VILLE_TO_DEPT
    3. Sélectionner uniquement les colonnes de pollens clés
    4. Agréger au mois → moyenne mensuelle par département
    5. Calculer des indicateurs synthétiques de risque pollinique

    COLONNES PRODUITES :
    ────────────────────
    dept                → code département
    annee_mois          → période (ex: "2021-03")
    pollen_ambrosia_moy → concentration moyenne ambroisie (grains/m³)
    pollen_betula_moy   → concentration moyenne bouleau
    pollen_graminees_moy→ concentration moyenne graminées
    pollen_alnus_moy    → concentration moyenne aulne
    pollen_artemisia_moy→ concentration moyenne armoise
    pollen_global_max   → concentration max tous pollens confondus
    nb_jours_eleve      → nb jours avec au moins un pollen > 50 grains/m³

    CLÉ PRIMAIRE : dept × annee_mois
    """

    pollen_dir = RAW_DIR / "BDD_daily_2020_2025"
    if not pollen_dir.exists():
        print(f"⚠️  Dossier manquant : {pollen_dir}")
        return pd.DataFrame()

    dfs = []
    nb_ok = 0
    nb_err = 0

    fichiers = sorted(pollen_dir.glob("*.xls"))
    print(f"Chargement de {len(fichiers)} fichiers RNSA...")

    for fpath in fichiers:
        # Extraire ville et année depuis le nom de fichier
        match = re.search(r'Particle_Extract_(.+?)_(\d{4})-\d{2}-\d{2}', fpath.name)
        if not match:
            continue

        ville = match.group(1)
        annee = int(match.group(2))

        # Rattacher au département
        dept = VILLE_TO_DEPT.get(ville)
        if dept is None:
            continue  # ville inconnue ou hors France (ex: ANDORRA)

        try:
            df = pd.read_excel(fpath, engine="xlrd")

            # Première colonne = date
            col_date = df.columns[0]
            df = df.rename(columns={col_date: "date"})
            df["date"] = pd.to_datetime(df["date"], errors="coerce")
            df = df.dropna(subset=["date"])

            # Filtrage temporel
            df = df[(df["date"].dt.year >= ANNEE_DEBUT) &
                    (df["date"].dt.year <= ANNEE_FIN)]

            if df.empty:
                continue

            df["annee_mois"] = df["date"].dt.to_period("M").astype(str)
            df["dept"]       = dept

            # Identifier les colonnes de pollens clés disponibles
            cols_pollen = {}
            for col in df.columns:
                col_upper = col.upper()
                for nom_brut, nom_propre in POLLEN_CLES.items():
                    if nom_brut in col_upper:
                        df[col] = pd.to_numeric(df[col], errors="coerce")
                        cols_pollen[col] = f"pollen_{nom_propre}"
                        break

            if not cols_pollen:
                continue

            # Renommer les colonnes de pollens
            df = df.rename(columns=cols_pollen)
            cols_utiles = ["dept", "annee_mois"] + list(set(cols_pollen.values()))
            df = df[[c for c in cols_utiles if c in df.columns]]

            # Indicateur : nb jours avec pollen élevé (> 50 grains/m³)
            pollen_cols_vals = [c for c in df.columns
                                if c.startswith("pollen_")]
            if pollen_cols_vals:
                df["jour_eleve"] = (df[pollen_cols_vals].max(axis=1) > 50).astype(int)
                df["pollen_global_max"] = df[pollen_cols_vals].max(axis=1)

            dfs.append(df)
            nb_ok += 1

        except Exception as e:
            nb_err += 1

    if not dfs:
        print("❌ Aucun fichier traité")
        return pd.DataFrame()

    print(f"✅ {nb_ok} fichiers chargés | {nb_err} erreurs")

    # Concaténer tous les fichiers
    df_all = pd.concat(dfs, ignore_index=True)
    print(f"Total lignes brutes : {len(df_all):,}")

    # Agrégation mensuelle par département
    # Plusieurs villes peuvent appartenir au même département
    # → on fait la moyenne de toutes les stations du même dept × mois
    agg_dict = {}
    for col in df_all.columns:
        if col.startswith("pollen_") and col != "pollen_global_max":
            agg_dict[f"{col}_moy"] = (col, "mean")
    if "pollen_global_max" in df_all.columns:
        agg_dict["pollen_global_max"] = ("pollen_global_max", "max")
    if "jour_eleve" in df_all.columns:
        agg_dict["nb_jours_eleve"] = ("jour_eleve", "max")  # max parmi les villes du dept (pas sum qui gonfle le chiffre)

    df_agg = (
        df_all.groupby(["dept", "annee_mois"])
        .agg(**agg_dict)
        .reset_index()
    )

    # Arrondi
    for col in df_agg.select_dtypes(include="float").columns:
        df_agg[col] = df_agg[col].round(2)

    print(f"\n✅ dim_pollen : {df_agg.shape[0]:,} lignes × {df_agg.shape[1]} colonnes")
    print(f"Période      : {df_agg['annee_mois'].min()} → {df_agg['annee_mois'].max()}")
    print(f"Départements : {df_agg['dept'].nunique()} couverts")
    print(f"Colonnes     : {list(df_agg.columns)}")

    # Couverture par département
    depts_couverts = set(df_agg["dept"].unique())
    depts_manquants = set(DEPTS) - depts_couverts
    print(f"\n⚠️  {len(depts_manquants)} depts sans données pollen :")
    print(f"   {sorted(depts_manquants)}")
    print("   → Ces depts seront NaN dans df_model (normal)")

    return df_agg


# ══════════════════════════════════════════════════════════════════════════════
# EXÉCUTION
# ══════════════════════════════════════════════════════════════════════════════
print("Construction de dim_pollen...")
dim_pollen = build_dim_pollen()

if not dim_pollen.empty:
    dim_pollen.to_parquet(TABLES_DIR / "dim_pollen.parquet", index=False)
    print(f"\n✅ Sauvegardé → data/tables/dim_pollen.parquet")
    display(dim_pollen.head(10))

Construction de dim_pollen...
Chargement de 253 fichiers RNSA...
✅ 154 fichiers chargés | 0 erreurs
Total lignes brutes : 56,047

✅ dim_pollen : 1,686 lignes × 11 colonnes
Période      : 2020-01 → 2024-06
Départements : 49 couverts
Colonnes     : ['dept', 'annee_mois', 'pollen_platane_moy', 'pollen_graminees_moy', 'pollen_ambrosia_moy', 'pollen_artemisia_moy', 'pollen_alnus_moy', 'pollen_betula_moy', 'pollen_urticacees_moy', 'pollen_global_max', 'nb_jours_eleve']

⚠️  47 depts sans données pollen :
   ['02', '04', '05', '08', '09', '11', '12', '17', '19', '23', '27', '28', '2B', '32', '36', '39', '40', '41', '42', '43', '46', '48', '50', '53', '55', '58', '59', '60', '61', '62', '63', '64', '65', '66', '70', '71', '77', '78', '82', '85', '88', '89', '90', '91', '93', '94', '95']
   → Ces depts seront NaN dans df_model (normal)

✅ Sauvegardé → data/tables/dim_pollen.parquet


,dept,annee_mois,pollen_platane_moy,pollen_graminees_moy,pollen_ambrosia_moy,pollen_artemisia_moy,pollen_alnus_moy,pollen_betula_moy,pollen_urticacees_moy,pollen_global_max,nb_jours_eleve
0,01,2020-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
1,01,2020-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
2,01,2020-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
3,01,2020-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
4,01,2020-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
5,01,2020-06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
6,01,2020-07,0.0,3.30,1.33,1.13,0.0,0.0,15.68,45.51,0
7,01,2020-08,0.0,2.89,8.99,1.74,0.0,0.0,17.08,63.13,1
8,01,2020-09,0.0,2.22,37.19,0.63,0.0,0.0,5.04,110.82,1
9,01,2020-10,0.0,0.94,1.71,0.08,0.0,0.0,0.52,9.75,0


---
## 6. 🔍 Audit de qualité — toutes les tables

In [32]:
TABLES = {
    "fact_urgences"   : TABLES_DIR / "fact_urgences.parquet",
    "dim_temps"       : TABLES_DIR / "dim_temps.parquet",
    "dim_meteo"       : TABLES_DIR / "dim_meteo.parquet",
    "dim_qualite_air" : TABLES_DIR / "dim_qualite_air.parquet",
    "dim_pollen"      : TABLES_DIR / "dim_pollen.parquet",
    "dim_geo_pop"     : TABLES_DIR / "dim_geo_pop.parquet",
    "dim_csp"         : TABLES_DIR / "dim_csp.parquet",
}

print("=" * 65)
print("AUDIT DES TABLES")
print("=" * 65)

for nom, fpath in TABLES.items():
    if not fpath.exists():
        print(f"\n❌ {nom} — fichier absent")
        continue

    df = pd.read_parquet(fpath)
    print(f"\n📋 {nom}")
    print(f"   Shape      : {df.shape[0]:>6,} lignes × {df.shape[1]} colonnes")

    # Clé primaire
    pk_candidates = [c for c in ["dept", "annee_mois", "annee"] if c in df.columns]
    if pk_candidates:
        dupes = df.duplicated(subset=pk_candidates).sum()
        print(f"   PK ({', '.join(pk_candidates)}) : {dupes} doublons {'✅' if dupes == 0 else '⚠️'}")

    # Taux de manquants
    missing = df.isnull().mean() * 100
    problemes = missing[missing > 5].round(1)
    if problemes.empty:
        print(f"   Manquants  : aucune colonne > 5% ✅")
    else:
        print(f"   Manquants  :")
        for col, pct in problemes.items():
            print(f"     {col:<35} {pct}%")

print("\n" + "=" * 65)

AUDIT DES TABLES

📋 fact_urgences
   Shape      :  6,912 lignes × 11 colonnes
   PK (dept, annee_mois) : 0 doublons ✅
   Manquants  :
     taux_sos_allergie                   53.5%
     taux_sos_asthme                     53.5%
     taux_sos_bronchiolite               53.6%

📋 dim_temps
   Shape      :     72 lignes × 13 colonnes
   PK (annee_mois, annee) : 0 doublons ✅
   Manquants  : aucune colonne > 5% ✅

📋 dim_meteo
   Shape      :  2,883 lignes × 8 colonnes
   PK (dept, annee_mois) : 0 doublons ✅
   Manquants  : aucune colonne > 5% ✅

📋 dim_qualite_air
   Shape      :  2,518 lignes × 9 colonnes
   PK (dept, annee_mois) : 0 doublons ✅
   Manquants  :
     o3_moy                              11.0%
     so2_moy                             44.3%

📋 dim_pollen
   Shape      :  1,686 lignes × 11 colonnes
   PK (dept, annee_mois) : 0 doublons ✅
   Manquants  :
     pollen_platane_moy                  41.4%
     pollen_graminees_moy                32.1%
     pollen_ambrosia_moy           

---
## 7. 🔗 Vue analytique — `df_model` (jointure pour la modélisation)

Pour XGBoost ou LSTM, on a besoin d'un DataFrame plat.  
Cette cellule joint toutes les tables en un seul `df_model` sans toucher aux tables sources.

In [33]:
def build_model_view() -> pd.DataFrame:
    """
    Crée la vue analytique plate pour la modélisation.
    Jointure : fact_urgences ← dim_temps ← dim_meteo ← dim_geo_pop ← dim_csp
    """
    if not (TABLES_DIR / "fact_urgences.parquet").exists():
        print("⚠️  fact_urgences.parquet manquant — exécutez d'abord les cellules 1 à 5")
        return pd.DataFrame()

    df = pd.read_parquet(TABLES_DIR / "fact_urgences.parquet")

    # Jointure dim_temps (annee_mois → enrichissement temporel)
    if (TABLES_DIR / "dim_temps.parquet").exists():
        dt = pd.read_parquet(TABLES_DIR / "dim_temps.parquet")
        df = df.merge(dt, on="annee_mois", how="left")
        print("✅ dim_temps joint")

    # Jointure dim_meteo (dept × annee_mois)
    if (TABLES_DIR / "dim_meteo.parquet").exists():
        dm = pd.read_parquet(TABLES_DIR / "dim_meteo.parquet")
        df = df.merge(dm, on=["dept", "annee_mois"], how="left")
        print("✅ dim_meteo joint")

    # Jointure dim_geo_pop (dept)
    if (TABLES_DIR / "dim_geo_pop.parquet").exists():
        dg = pd.read_parquet(TABLES_DIR / "dim_geo_pop.parquet")
        df = df.merge(dg, on="dept", how="left")
        print("✅ dim_geo_pop joint")

    # Jointure dim_csp (dept)
    if (TABLES_DIR / "dim_csp.parquet").exists():
        dc = pd.read_parquet(TABLES_DIR / "dim_csp.parquet")
        df = df.merge(dc, on="dept", how="left")
        print("✅ dim_csp joint")

    # Jointure dim_qualite_air (dept × annee_mois)
    if (TABLES_DIR / "dim_qualite_air.parquet").exists():
        dqa = pd.read_parquet(TABLES_DIR / "dim_qualite_air.parquet")
        df = df.merge(dqa, on=["dept", "annee_mois"], how="left")
        print("✅ dim_qualite_air joint")

    # Jointure dim_pollen (dept × annee_mois)
    if (TABLES_DIR / "dim_pollen.parquet").exists():
        dp = pd.read_parquet(TABLES_DIR / "dim_pollen.parquet")
        df = df.merge(dp, on=["dept", "annee_mois"], how="left")
        print("✅ dim_pollen joint")

    # Vérification de couverture : les colonnes de taux existent-elles ?
    cols_taux_urgences = [c for c in df.columns if c.startswith("taux_urgences_")]
    if not cols_taux_urgences:
        print("⚠️  Aucune colonne taux_urgences_* trouvée dans fact_urgences")

    # Résumé du taux de remplissage par groupe de variables
    groupes = {
        "Y (urgences)": [c for c in df.columns if c.startswith("taux_urgences_")],
        "Météo":        [c for c in df.columns if c in ["temp_moy","temp_min","temp_max","humidite_moy","vent_moy"]],
        "Air quality":  [c for c in df.columns if c.startswith("atmo_") or (c.endswith("_moy") and c[:2] in ["pm","no","o3"])],
        "Pollen":       [c for c in df.columns if c.startswith("pollen_") or c == "nb_semaines_elevee"],
        "Démographie":  [c for c in df.columns if c in ["pop_totale","part_seniors","part_jeunes","tx_urbain"]],
    }
    print("\nTaux de remplissage par groupe :")
    for groupe, cols in groupes.items():
        if cols:
            pct = df[cols].notna().mean().mean() * 100
            bar = '█' * int(pct // 10) + '░' * (10 - int(pct // 10))
            print(f"  {groupe:<18} {bar} {pct:.0f}%")

    print(f"\n📦 df_model : {df.shape[0]:,} lignes × {df.shape[1]} colonnes")
    return df


df_model = build_model_view()

if not df_model.empty:
    df_model.to_parquet(TABLES_DIR / "df_model.parquet", index=False)
    df_model.to_csv(TABLES_DIR / "df_model.csv", index=False)
    print(f"\n✅ Vue analytique sauvegardée : {TABLES_DIR / 'df_model.parquet'}")
    print("\nColonnes disponibles pour la modélisation :")
    print(list(df_model.columns))
    display(df_model.head(5))

✅ dim_temps joint
✅ dim_meteo joint
✅ dim_geo_pop joint
✅ dim_csp joint
✅ dim_qualite_air joint
✅ dim_pollen joint

Taux de remplissage par groupe :
  Y (urgences)       █████████░ 100%
  Météo              ████░░░░░░ 42%
  Air quality        ███░░░░░░░ 35%
  Pollen             █░░░░░░░░░ 15%
  Démographie        ██████████ 100%

📦 df_model : 6,912 lignes × 58 colonnes

✅ Vue analytique sauvegardée : data/tables/df_model.parquet

Colonnes disponibles pour la modélisation :
['dept', 'annee_mois', 'taux_urgences_allergie', 'taux_hosp_allergie', 'taux_sos_allergie', 'taux_urgences_asthme', 'taux_hosp_asthme', 'taux_sos_asthme', 'taux_urgences_bronchiolite', 'taux_hosp_bronchiolite', 'taux_sos_bronchiolite', 'annee', 'mois', 'trimestre', 'semestre', 'sin_mois', 'cos_mois', 'est_hiver', 'est_printemps', 'est_ete', 'est_automne', 'saison_pollen', 'flag_covid', 'temp_moy', 'temp_min', 'temp_max', 'humidite_moy', 'vent_moy', 'precip_total', 'pop_totale', 'part_seniors', 'part_jeunes', 'tx_urba

,dept,annee_mois,taux_urgences_allergie,taux_hosp_allergie,taux_sos_allergie,taux_urgences_asthme,taux_hosp_asthme,taux_sos_asthme,taux_urgences_bronchiolite,taux_hosp_bronchiolite,...,nb_jours_pm10_eleve,pollen_platane_moy,pollen_graminees_moy,pollen_ambrosia_moy,pollen_artemisia_moy,pollen_alnus_moy,pollen_betula_moy,pollen_urticacees_moy,pollen_global_max,nb_jours_eleve
0,01,2020-01,815.39,245.12,NaN,402.03,878.26,NaN,23287.66,44861.11,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
1,01,2020-02,722.06,228.66,NaN,462.67,437.93,NaN,10169.56,25875.35,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
2,01,2020-03,377.48,193.97,NaN,805.16,841.07,NaN,9939.19,14230.77,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
3,01,2020-04,500.03,98.04,NaN,574.89,863.81,NaN,0.00,0.00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
4,01,2020-05,568.77,311.94,NaN,522.41,552.69,NaN,0.00,0.00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0


---
## 📁 Récapitulatif des fichiers produits

```
data/
├── raw/                                      ← Fichiers bruts téléchargés manuellement
│   ├── allergie_urgences.csv                 ← Santé Publique France
│   ├── asthme_urgences.csv
│   ├── bronchiolite_urgences.csv
│   ├── synop/
│   │   ├── synop_2020.csv.gz                 ← Météo France SYNOP (annuel)
│   │   ├── synop_2021.csv.gz
│   │   ├── synop_2022.csv.gz
│   │   ├── synop_2023.csv.gz
│   │   ├── synop_2024.csv.gz
│   │   └── synop_2025.csv.gz
│   ├── OBSERVATIONS_liste_stations_SYNOP_OMM.csv
│   ├── AASQA/
│   │   ├── FR_E2_2021-01-01.csv              ← Qualité de l'air AASQA (mensuel)
│   │   └── ... (60 fichiers, 2021-2025)
│   ├── BDD_daily_2020_2025/
│   │   ├── Particle_Extract_AGEN_2020-...xls ← Pollen RNSA (253 fichiers)
│   │   └── ...
│   └── geo_pop/
│       ├── insee_demographie.csv
│       ├── insee_urbanisation.xlsx
│       ├── insee_csp.xlsx
│       ├── ameli_acces_soins_2020.xls
│       ├── ameli_acces_soins_2021.xls
│       ├── ameli_acces_soins_2022.xls
│       ├── ameli_acces_soins_2023.xls
│       └── ameli_acces_soins_2024.xls
│
└── tables/                                   ← Tables produites par ce notebook
├── stations_dept.parquet                 ← Correspondance station SYNOP → dept
├── dim_temps.parquet                     ← 72 mois, variables temporelles + saisonnalité
├── fact_urgences.parquet                 ← 6 912 lignes, 3 pathologies × 3 indicateurs
├── dim_meteo.parquet                     ← 2 883 lignes, 41 depts × 72 mois
├── dim_qualite_air.parquet               ← 2 518 lignes, 42 depts × 60 mois (2021-2025)
├── dim_pollen.parquet                    ← 1 686 lignes, 49 depts × 3 années
├── dim_geo_pop.parquet                   ← 96 depts, démographie + urbanisation + soins
├── dim_csp.parquet                       ← 96 depts, 7 catégories socioprofessionnelles
├── df_model.parquet                      ← 6 912 lignes × 58 colonnes (vue analytique)
└── df_model.csv                          ← Idem, format CSV
```
### Ordre d'exécution
| Étape | Table | Données requises | Lignes produites |
|---|---|---|---|
| 0 | Configuration | — | — |
| 1 | `dim_temps` | Aucune (calculé) | 72 |
| 2 | `fact_urgences` | allergie / asthme / bronchiolite CSV | 6 912 |
| 3 | `dim_meteo` | synop_YYYY.csv.gz + liste stations | 2 883 |
| 4 | `dim_geo_pop` | insee_demographie + urbanisation + ameli | 96 |
| 5 | `dim_csp` | insee_csp.xlsx (feuille "dep") | 96 |
| 6A | `dim_qualite_air` | AASQA/FR_E2_*.csv | 2 518 |
| 6B | `dim_pollen` | BDD_daily_2020_2025/*.xls | 1 686 |
| 6 | Audit qualité | Toutes les tables ci-dessus | — |
| 7 | `df_model` | Toutes les tables ci-dessus | 6 912 × 58 col |

### Limites connues
| Variable | Statut | Raison |
|---|---|---|
| Qualité de l'air 2020 | ❌ Absent | Données AASQA non disponibles avant 2021 |
| Pollen 2022, 2024, 2025 | ❌ Absent | Non publiés par le RNSA dans BDD_daily |
| Météo (55 depts) | ⚠️ NaN | Aucune station SYNOP dans ces départements |
| SO₂ | ⚠️ 56% | Stations de mesure SO₂ peu nombreuses |